In [17]:
import pandas as pd
import numpy as np

In [18]:
ts_df = pd.read_csv("timeseries.csv")

In [19]:
rogue_header_mask = (ts_df.astype(str) == ts_df.columns).any(axis=1)

# 2. Filter the dataframe to keep only rows that are NOT rogue headers
ts_df = ts_df[~rogue_header_mask].copy()

ts_df['scraped_at'] = pd.to_datetime(ts_df['scraped_at'])

In [20]:
videos_df = pd.read_csv("videos.csv")

In [21]:
# 1. Reassign the converted dates back to the specific column, not the whole dataframe
videos_df['published_at'] = pd.to_datetime(videos_df['published_at'])

# 2. Use quotes around the column name in the filter
threshold_date = pd.to_datetime("2026-07-27 02:06:50.602131+00")
true_day_0_vids = videos_df[videos_df['published_at'] >= threshold_date].index
late_discovery_vids = videos_df[videos_df['published_at'] < threshold_date].index
print(f"Total videos: {len(videos_df)}")
print(f"Late discovery videos: {len(late_discovery_vids)}")

Total videos: 10747
Late discovery videos: 4463


In [22]:
videos_df['published_at'] = pd.to_datetime(videos_df['published_at'])
videos_df['created_at'] = pd.to_datetime(videos_df['created_at'])

threshold_date = pd.to_datetime("2026-07-27 02:06:50.602131+00")

true_day_0 = videos_df[videos_df['published_at'] >= threshold_date].copy()

true_day_0['discovery_lag_h'] = (
    true_day_0['created_at'] - true_day_0['published_at']
).dt.total_seconds() / 3600

print("Discovery lag (hours) for true-day-0 subset:")
print(true_day_0['discovery_lag_h'].describe())
print("\nWithin 15 min:", (true_day_0['discovery_lag_h'] <= 0.25).sum(), "/", len(true_day_0))
print("Within 2h:     ", (true_day_0['discovery_lag_h'] <= 2).sum(), "/", len(true_day_0))
print("Beyond 2h:     ", (true_day_0['discovery_lag_h'] > 2).sum(), "/", len(true_day_0))

# if a meaningful tail is beyond 2h, tighten the filter to combine both conditions
still_late = true_day_0[true_day_0['discovery_lag_h'] > 2]
if len(still_late) > 0:
    print(f"\n{len(still_late)} videos still discovered >2h late despite passing the date filter:")
    print(still_late[['video_id', 'published_at', 'created_at', 'discovery_lag_h']].sort_values(
        'discovery_lag_h', ascending=False).head(10))

Discovery lag (hours) for true-day-0 subset:
count    6284.000000
mean        3.698124
std        19.724657
min      -369.422825
25%         1.574862
50%         4.633049
75%         8.636095
max        13.993424
Name: discovery_lag_h, dtype: float64

Within 15 min: 579 / 6284
Within 2h:      1856 / 6284
Beyond 2h:      4428 / 6284

4428 videos still discovered >2h late despite passing the date filter:
          video_id              published_at                       created_at  \
10434  Z1Kkz8Ot7ao 2026-08-02 13:52:21+00:00 2026-08-03 03:51:57.327570+00:00   
1891   9NqOSlYWeI4 2026-08-02 14:00:33+00:00 2026-08-03 03:51:57.327570+00:00   
6071   M7e1GvAzoyM 2026-08-02 14:01:36+00:00 2026-08-03 03:51:57.327570+00:00   
6864   OISn13iX7-Y 2026-08-02 14:05:48+00:00 2026-08-03 03:51:57.327570+00:00   
257    -MPoa8Miv-4 2026-08-01 14:07:51+00:00 2026-08-02 03:50:07.795991+00:00   
7804   RA_rS048y-Y 2026-08-02 14:16:47+00:00 2026-08-03 03:51:57.327570+00:00   
9453   w88y2nBWzWc 2026-08-

In [23]:
import pandas as pd

videos_df['published_at'] = pd.to_datetime(videos_df['published_at'])
videos_df['created_at'] = pd.to_datetime(videos_df['created_at'])
videos_df['discovery_lag_h'] = (
    (videos_df['created_at'] - videos_df['published_at']).dt.total_seconds() / 3600
)

neg = videos_df[videos_df['discovery_lag_h'] < 0].copy()
print(f"Total negative-lag videos: {len(neg)} / {len(videos_df)}")
print(neg['discovery_lag_h'].describe())

# --- 1. worst offenders, full detail ---
worst = neg.sort_values('discovery_lag_h').head(15)
cols = ['video_id', 'channel_id', 'title', 'published_at', 'created_at',
        'discovery_lag_h', 'duration', 'category_id']
print("\n--- Most negative lag videos ---")
print(worst[cols].to_string())

# --- 2. are these premieres / livestreams? check title for keywords ---
keywords = ['premiere', 'live', 'streaming', 'stream']
neg['title_lower'] = neg['title'].str.lower().fillna('')
neg['looks_like_live'] = neg['title_lower'].str.contains('|'.join(keywords))
print(f"\nNegative-lag videos with live/premiere keywords in title: {neg['looks_like_live'].sum()} / {len(neg)}")

# --- 3. are negative-lag videos concentrated in specific channels? ---
by_channel = neg.groupby('channel_id').size().sort_values(ascending=False)
print(f"\nDistinct channels with negative-lag videos: {len(by_channel)}")
print("Top channels by count of negative-lag videos:")
print(by_channel.head(10))

# --- 4. does this channel have OTHER videos with normal (positive, small) lag? ---
# if yes -> channel was legitimately tracked early, this is fine
# if no -> every video from this channel has bad lag, suspect systemic issue
sample_channel = by_channel.index[0]
channel_videos = videos_df[videos_df['channel_id'] == sample_channel][
    ['video_id', 'published_at', 'created_at', 'discovery_lag_h']
].sort_values('published_at')
print(f"\nAll videos from channel {sample_channel} (top offender), sorted by publish date:")
print(channel_videos.to_string())

# --- 5. cross-check against actual first scrape time (ground truth) ---
ts_df['scraped_at'] = pd.to_datetime(ts_df['scraped_at'])
first_scrape = (
    ts_df.sort_values('scraped_at')
    .drop_duplicates('video_id', keep='first')
    [['video_id', 'scraped_at']]
    .rename(columns={'scraped_at': 'first_scrape_at'})
)
neg_check = neg.merge(first_scrape, on='video_id', how='left')
neg_check['true_first_age_h'] = (
    (neg_check['first_scrape_at'] - neg_check['published_at']).dt.total_seconds() / 3600
)
print("\n--- For negative-lag videos: true first scrape age (hours since publish) ---")
print(neg_check['true_first_age_h'].describe())
print("\nIf this looks reasonable (small positive number), created_at is just a poor proxy")
print("and true_first_age_h should be used instead everywhere, not created_at.")

Total negative-lag videos: 359 / 10747
count    359.000000
mean     -26.051553
std       75.161390
min     -369.422825
25%      -11.023975
50%       -5.036176
75%       -1.180407
max       -0.002417
Name: discovery_lag_h, dtype: float64

--- Most negative lag videos ---
          video_id                channel_id                                                                                                 title              published_at                       created_at  discovery_lag_h duration  category_id
6840   OFV0Y5rZt0Y  UCjK3LREYZUrNKnk56Prhe4w  🕯️The Unraveling🕯️- The Bodhisattva's Contemplation Vocal Forward Post Rock Mix(බෝධිමූල මංගල්‍යය)🕯️🎵 2026-08-11 11:30:41+00:00 2026-07-27 02:05:18.831017+00:00      -369.422825  PT6M45S         29.0
925    3peBZX907Jk  UCjK3LREYZUrNKnk56Prhe4w  ✨The Ascent from the Affixed Mind✨Greed,Craving, Liberation- A Cyber-Gothic Techno Liturgy🩸🕸️(තණ්හා) 2026-08-11 11:28:31+00:00 2026-07-27 02:05:18.831017+00:00      -369.386714   PT6M1S        

In [24]:
HORIZON_H = 120
DAY0_THRESHOLD_H = 2.0   # adjust after checking yield at a few thresholds

videos_df['published_at'] = pd.to_datetime(videos_df['published_at'])
ts_df['scraped_at'] = pd.to_datetime(ts_df['scraped_at'])

# --- true first/last scrape age per video ---
ts_sorted = ts_df.sort_values('scraped_at')
first_scrape = (
    ts_sorted.drop_duplicates('video_id', keep='first')
    [['video_id', 'scraped_at']].rename(columns={'scraped_at': 'first_scrape_at'})
)
last_scrape = (
    ts_sorted.drop_duplicates('video_id', keep='last')
    [['video_id', 'scraped_at']].rename(columns={'scraped_at': 'last_scrape_at'})
)

vids = videos_df.merge(first_scrape, on='video_id', how='left') \
                .merge(last_scrape, on='video_id', how='left')

vids['true_first_age_h'] = (vids['first_scrape_at'] - vids['published_at']).dt.total_seconds() / 3600
vids['true_last_age_h']  = (vids['last_scrape_at']  - vids['published_at']).dt.total_seconds() / 3600

# --- check yield at a few day-0 thresholds before committing ---
for th in [0.25, 0.5, 1.0, 2.0]:
    n = (vids['true_first_age_h'] <= th).sum()
    print(f"Day-0 threshold {th}h: {n} videos eligible")

# --- apply filters ---
has_day0   = vids['true_first_age_h'] <= DAY0_THRESHOLD_H
has_horizon = vids['true_last_age_h'] >= HORIZON_H

print(f"\nStatus value counts:\n{vids['status'].value_counts()}")
is_active_enough = vids['status'] != 'deleted'   # adjust based on actual status values printed above

eligible = vids[has_day0 & has_horizon & is_active_enough].copy()

print(f"\nDay-0 eligible: {has_day0.sum()}")
print(f"Horizon eligible: {has_horizon.sum()}")
print(f"Status OK: {is_active_enough.sum()}")
print(f"Final eligible (all three): {len(eligible)} / {len(videos_df)}")

eligible.to_csv('eligible_videos.csv', index=False)

Day-0 threshold 0.25h: 494 videos eligible
Day-0 threshold 0.5h: 703 videos eligible
Day-0 threshold 1.0h: 1020 videos eligible
Day-0 threshold 2.0h: 1675 videos eligible

Status value counts:
status
active     10606
deleted      141
Name: count, dtype: int64

Day-0 eligible: 1675
Horizon eligible: 8614
Status OK: 10606
Final eligible (all three): 1250 / 10747


In [25]:
HORIZON_H = 120

# grid: 5-min resolution 0-2h, hourly 2-120h
grid_dense = np.arange(0, 2 + 1/12, 5/60)          # 0, 0.083, 0.167, ..., 2.0
grid_sparse = np.arange(3, HORIZON_H + 1, 1)        # 3, 4, ..., 120
GRID_H = np.unique(np.concatenate([grid_dense, grid_sparse]))
print(f"Grid points: {len(GRID_H)}")

# --- merge ---
eligible['published_at'] = pd.to_datetime(eligible['published_at'])
ts_df['scraped_at'] = pd.to_datetime(ts_df['scraped_at'])

ts = ts_df[ts_df['video_id'].isin(eligible['video_id'])].copy()
ts = ts.merge(eligible[['video_id', 'published_at']], on='video_id', how='inner')
ts['age_h'] = (ts['scraped_at'] - ts['published_at']).dt.total_seconds() / 3600
ts = ts[(ts['age_h'] >= 0) & (ts['age_h'] <= HORIZON_H * 1.02)]
ts = ts.sort_values(['video_id', 'age_h']).drop_duplicates(['video_id', 'age_h'])

# --- resample each video onto GRID_H via log-linear interpolation ---
def resample_video(group):
    ages = group['age_h'].values
    views = group['view_count'].values.astype(float)

    # need at least 2 points to interpolate
    if len(ages) < 2:
        return None

    log_views = np.log1p(views)
    # clip grid to the video's actual observed range; extrapolation beyond
    # the last real point is not reliable, so cap grid at the video's max observed age
    max_age = ages.max()
    valid_grid = GRID_H[GRID_H <= max_age]
    if len(valid_grid) < 2:
        return None

    interp_log = np.interp(valid_grid, ages, log_views)
    interp_views = np.expm1(interp_log)

    row = pd.Series(index=[f"h{g:.3f}" for g in GRID_H], dtype=float)
    row[[f"h{g:.3f}" for g in valid_grid]] = interp_views
    return row

resampled = ts.groupby('video_id').apply(resample_video)
resampled = resampled.dropna(how='all')
print(f"\nVideos successfully resampled: {len(resampled)}")

# --- coverage check: how many grid points does each video actually have? ---
coverage = resampled.notna().sum(axis=1)
print("\nGrid coverage per video (out of", len(GRID_H), "points):")
print(coverage.describe())

# --- monotonicity fix (cumulative views can't decrease) ---
vals = resampled.values
vals_fixed = np.fmax.accumulate(np.nan_to_num(vals, nan=-np.inf), axis=1)
vals_fixed[np.isnan(vals)] = np.nan   # restore NaNs where we had no data
resampled_fixed = pd.DataFrame(vals_fixed, index=resampled.index, columns=resampled.columns)

resampled_fixed.to_csv('resampled_corpus.csv')
print(f"\nSaved resampled_corpus.csv: {resampled_fixed.shape}")

Grid points: 144


C:\Users\ASUS\AppData\Local\Temp\ipykernel_21616\2260043370.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  resampled = ts.groupby('video_id').apply(resample_video)



Videos successfully resampled: 1250

Grid coverage per video (out of 144 points):
count    1250.0
mean      144.0
std         0.0
min       144.0
25%       144.0
50%       144.0
75%       144.0
max       144.0
dtype: float64

Saved resampled_corpus.csv: (1250, 144)


In [26]:
import pandas as pd
import numpy as np

# how many REAL scrapes does each video actually have (before resampling)?
real_counts = ts.groupby('video_id').size()
print("Real scrape count per video (raw, before interpolation):")
print(real_counts.describe())

# largest gap between consecutive real scrapes, per video
ts_sorted = ts.sort_values(['video_id', 'age_h'])
ts_sorted['gap_h'] = ts_sorted.groupby('video_id')['age_h'].diff()
max_gap = ts_sorted.groupby('video_id')['gap_h'].max()
print("\nLargest gap between real scrapes, per video (hours):")
print(max_gap.describe())
print("\nVideos with a gap > 6h somewhere in their series:", (max_gap > 6).sum())
print("Videos with a gap > 12h somewhere in their series:", (max_gap > 12).sum())
print("Videos with a gap > 24h somewhere in their series:", (max_gap > 24).sum())

Real scrape count per video (raw, before interpolation):
count    1250.0000
mean      118.0104
std         4.2166
min        99.0000
25%       115.0000
50%       118.0000
75%       121.0000
max       131.0000
dtype: float64

Largest gap between real scrapes, per video (hours):
count    1250.000000
mean        1.334386
std         0.737001
min         1.087078
25%         1.095353
50%         1.253174
75%         1.563716
max        23.995635
Name: gap_h, dtype: float64

Videos with a gap > 6h somewhere in their series: 2
Videos with a gap > 12h somewhere in their series: 1
Videos with a gap > 24h somewhere in their series: 0


In [27]:
bad_gap_videos = max_gap[max_gap > 6].index.tolist()
print(f"Dropping {len(bad_gap_videos)} videos with gaps > 6h: {bad_gap_videos}")

resampled_fixed = resampled_fixed.drop(index=bad_gap_videos, errors='ignore')
eligible_final = eligible[~eligible['video_id'].isin(bad_gap_videos)]

print(f"Final corpus: {len(resampled_fixed)} videos")

Dropping 2 videos with gaps > 6h: ['YhIW06P-RBA', 'tmHNkqCqJEM']
Final corpus: 1248 videos


In [28]:
n_channels = eligible_final['channel_id'].nunique()
print(f"Distinct channels: {n_channels} / {eligible['channel_id'].nunique()} total tracked")

per_channel = eligible_final.groupby('channel_id').size().sort_values(ascending=False)
print("\nVideos per channel:")
print(per_channel.describe())
print(f"\nChannels with only 1 video: {(per_channel == 1).sum()}")
print(f"Channels with 5+ videos:    {(per_channel >= 5).sum()}")
print(f"Channels with 10+ videos:   {(per_channel >= 10).sum()}")

Distinct channels: 37 / 37 total tracked

Videos per channel:
count     37.00000
mean      33.72973
std       91.65262
min        1.00000
25%        2.00000
50%        3.00000
75%       11.00000
max      409.00000
dtype: float64

Channels with only 1 video: 6
Channels with 5+ videos:    15
Channels with 10+ videos:   10


In [29]:
ps = pd.read_csv("ps_data.csv")

In [30]:
ps

,id,published_at,channel_id,title,description,localized_title,localized_description,thumbnail_default,thumbnail_medium,thumbnail_high,...,day_21_comments,day_22_comments,day_23_comments,day_24_comments,day_25_comments,day_26_comments,day_27_comments,day_28_comments,day_29_comments,day_30_comments
0,DtxyWGbFxtM,2025-08-14 20:09:07,UCckltLEhFLv8Xz_lQhYfwmg,ජනපති අණින් නැවතූ සුළං බලාගාර ව්‍යාපෘතිය ගැන ඉ...,ජනපති අණින් නැවතූ සුළං බලාගාර ව්‍යාපෘතිය ගැන ඉ...,ජනපති අණින් නැවතූ සුළං බලාගාර ව්‍යාපෘතිය ගැන ඉ...,ජනපති අණින් නැවතූ සුළං බලාගාර ව්‍යාපෘතිය ගැන ඉ...,https://i.ytimg.com/vi/DtxyWGbFxtM/default.jpg,https://i.ytimg.com/vi/DtxyWGbFxtM/mqdefault.jpg,https://i.ytimg.com/vi/DtxyWGbFxtM/hqdefault.jpg,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,UBM9EHy0w3g,2025-08-08 14:00:08,UCckltLEhFLv8Xz_lQhYfwmg,උතුම් නිකිණි පුන් පොහෝ දිනයේ මියුගුණාරාම රජමහා...,උතුම් නිකිණි පුන් පොහෝ දිනයේ මියුගුණාරාම රජමහා...,උතුම් නිකිණි පුන් පොහෝ දිනයේ මියුගුණාරාම රජමහා...,උතුම් නිකිණි පුන් පොහෝ දිනයේ මියුගුණාරාම රජමහා...,https://i.ytimg.com/vi/UBM9EHy0w3g/default.jpg,https://i.ytimg.com/vi/UBM9EHy0w3g/mqdefault.jpg,https://i.ytimg.com/vi/UBM9EHy0w3g/hqdefault.jpg,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,j-JYUMbiSFA,2025-08-06 19:06:43,UCrBzMpy0C8kRGCfjRG-F__A,Maths Crew Live Stream,@Maths Crew is the educational YouTube Channel...,Maths Crew Live Stream,@Maths Crew is the educational YouTube Channel...,https://i.ytimg.com/vi/j-JYUMbiSFA/default_liv...,https://i.ytimg.com/vi/j-JYUMbiSFA/mqdefault_l...,https://i.ytimg.com/vi/j-JYUMbiSFA/hqdefault_l...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7rcMv1Q9Vq0,2025-09-15 03:26:31,UC2VV8PlsjYVQ_0KaB0QFOuQ,Apple picking 🍎@ Downey’s apple & strawberry f...,NaN,Apple picking 🍎@ Downey’s apple & strawberry f...,NaN,https://i.ytimg.com/vi/7rcMv1Q9Vq0/default.jpg,https://i.ytimg.com/vi/7rcMv1Q9Vq0/mqdefault.jpg,https://i.ytimg.com/vi/7rcMv1Q9Vq0/hqdefault.jpg,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,oeB62XusMes,2025-10-22 16:39:21,UCl2A-e1K2AZ-N1O7yuz5QNw,ලෝක ළමා දින සැමරුම 2025,NaN,ලෝක ළමා දින සැමරුම 2025,NaN,https://i.ytimg.com/vi/oeB62XusMes/default.jpg,https://i.ytimg.com/vi/oeB62XusMes/mqdefault.jpg,https://i.ytimg.com/vi/oeB62XusMes/hqdefault.jpg,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206615,Zl_9O2DtdpM,2026-08-14 05:46:05,UCCK3OZi788Ok44K97WAhLKQ,"ලාල්ටයි, නාමල්ටයි තණකොළ කවන්න බෝලඅත්ත සූදානම් ...",Ada Derana | ශ්‍රී ලංකාවේ විශ්වසනීය හා ප්‍රමුඛ...,"ලාල්ටයි, නාමල්ටයි තණකොළ කවන්න බෝලඅත්ත සූදානම් ...",Ada Derana | ශ්‍රී ලංකාවේ විශ්වසනීය හා ප්‍රමුඛ...,https://i.ytimg.com/vi/Zl_9O2DtdpM/default.jpg,https://i.ytimg.com/vi/Zl_9O2DtdpM/mqdefault.jpg,https://i.ytimg.com/vi/Zl_9O2DtdpM/hqdefault.jpg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
206616,h3Lkfd5RTJ4,2026-08-15 05:00:03,UCG_NNpbNwsczo7w3PTrhOoA,🧊 A giant canyon hidden inside the Greenland i...,🧊 A giant canyon hidden inside the Greenland i...,🧊 A giant canyon hidden inside the Greenland i...,🧊 A giant canyon hidden inside the Greenland i...,https://i.ytimg.com/vi/h3Lkfd5RTJ4/default.jpg,https://i.ytimg.com/vi/h3Lkfd5RTJ4/mqdefault.jpg,https://i.ytimg.com/vi/h3Lkfd5RTJ4/hqdefault.jpg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
206617,35ItcsdXGQk,2026-08-15 05:54:22,UCOtYyt7W5PmPnwQjWWF_Z-Q,මන් මොකක් හරි ඇහුවොත් තමයි ඇත්ත කියන්න බැරි.,මන් මොකක් හරි ඇහුවොත් තමයි ඇත්ත කියන්න බැරි.\...,මන් මොකක් හරි ඇහුවොත් තමයි ඇත්ත කියන්න බැරි.,මන් මොකක් හරි ඇහුවොත් තමයි ඇත්ත කියන්න බැරි.\...,https://i.ytimg.com/vi/35ItcsdXGQk/default.jpg,https://i.ytimg.com/vi/35ItcsdXGQk/mqdefault.jpg,https://i.ytimg.com/vi/35ItcsdXGQk/hqdefault.jpg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
206618,MlJrzBvA9sw,2026-08-15 05:56:41,UCCK3OZi788Ok44K97WAhLKQ,අද දෙරණ 12.00 මධ්‍යාහ්න පුවත් විකාශයේ සිරස්තල ...,Ada Derana | ශ්‍රී ලංකාවේ විශ්වසනීය හා ප්‍රමුඛ...,අද දෙරණ 12.00 මධ්‍යාහ්න පුවත් විකාශයේ සිරස්තල ...,Ada Derana | ශ්‍රී ලංකාවේ විශ්වසනීය හා ප්‍රමුඛ...,https://i.ytimg.com/vi/MlJrzBvA9sw/default.j

In [31]:
import pandas as pd
import numpy as np

# adjust variable name to whatever you load it as
df2 = ps.copy()

print(f"Total videos: {len(df2)}")
print(f"Distinct channels: {df2['channel_id'].nunique()}")

per_channel = df2.groupby('channel_id').size().sort_values(ascending=False)
print("\nVideos per channel:")
print(per_channel.describe())
print(f"\nChannels with only 1 video: {(per_channel == 1).sum()}")
print(f"Channels with 5+ videos:    {(per_channel >= 5).sum()}")
print(f"Channels with 10+ videos:   {(per_channel >= 10).sum()}")
print(f"Top channel's share of total: {per_channel.iloc[0]} / {len(df2)} = {per_channel.iloc[0]/len(df2):.1%}")

# --- missing data check across the day_N_views columns ---
day_view_cols = [f"day_{i}_views" for i in range(1, 31)]
missing_per_video = df2[day_view_cols].isna().sum(axis=1)
print("\nMissing day-columns per video (out of 30):")
print(missing_per_video.describe())
print(f"\nVideos with complete 30-day data: {(missing_per_video == 0).sum()}")
print(f"Videos with day_1 present: {df2['day_1_views'].notna().sum()}")
print(f"Videos with day_30 present: {df2['day_30_views'].notna().sum()}")

# --- monotonicity check (cumulative views should never decrease day to day) ---
vals = df2[day_view_cols].values.astype(float)
diffs = np.diff(vals, axis=1)
non_monotonic = np.nansum(diffs < 0, axis=1)
print(f"\nVideos with any day-to-day decrease: {(non_monotonic > 0).sum()}")

# --- overlap with our existing channels ---
our_channel_ids = set(videos_df['channel_id'].unique())
their_channel_ids = set(df2['channel_id'].unique())
overlap = our_channel_ids & their_channel_ids
new_channels = their_channel_ids - our_channel_ids

print(f"\nOur channels: {len(our_channel_ids)}")
print(f"Their channels: {len(their_channel_ids)}")
print(f"Overlapping channels: {len(overlap)}")
print(f"New channels we don't have: {len(new_channels)}")

# --- publish date range, for context ---
df2['published_at'] = pd.to_datetime(df2['published_at'])
print(f"\nPublish date range: {df2['published_at'].min()} to {df2['published_at'].max()}")

Total videos: 206620
Distinct channels: 1084

Videos per channel:
count     1084.000000
mean       190.608856
std       1836.728094
min          1.000000
25%          5.000000
50%         18.000000
75%         68.250000
max      46485.000000
dtype: float64

Channels with only 1 video: 118
Channels with 5+ videos:    821
Channels with 10+ videos:   705
Top channel's share of total: 46485 / 206620 = 22.5%

Missing day-columns per video (out of 30):
count    206620.000000
mean          1.590456
std           5.570948
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          30.000000
dtype: float64

Videos with complete 30-day data: 186500
Videos with day_1 present: 205733
Videos with day_30 present: 186500

Videos with any day-to-day decrease: 32589

Our channels: 100
Their channels: 1084
Overlapping channels: 13
New channels we don't have: 1071

Publish date range: 2025-08-06 01:16:25 to 2026-08-16 01:15:09


In [32]:
import pandas as pd
import numpy as np

day_view_cols = [f"day_{i}_views" for i in range(1, 31)]

# --- 1. who is the dominant channel? ---
top_channel_id = df2.groupby('channel_id').size().idxmax()
top_channel_videos = df2[df2['channel_id'] == top_channel_id]
print(f"Top channel: {top_channel_id}")
print(f"Video count: {len(top_channel_videos)}")
print(f"Sample titles:\n{top_channel_videos['title'].head(10).to_string()}")
print(f"\nCategory distribution:\n{top_channel_videos['category_id'].value_counts()}")
print(f"\nAvg duration: {top_channel_videos['video_duration'].mode()}")
print(f"Publish frequency: {len(top_channel_videos) / ((top_channel_videos['published_at'].max() - top_channel_videos['published_at'].min()).days + 1):.1f} videos/day")

# --- 2. magnitude of day-to-day decreases ---
vals = df2[day_view_cols].values.astype(float)
diffs = np.diff(vals, axis=1)

# relative drop size where a decrease occurs
neg_mask = diffs < 0
rel_drop = np.where(neg_mask, -diffs / np.maximum(vals[:, :-1], 1), np.nan)

print(f"\nRelative size of drops (as fraction of prior day's views), where decreases occur:")
print(pd.Series(rel_drop[neg_mask]).describe())

# how many videos have a SUBSTANTIAL drop (>5% of that day's views)?
big_drop_mask = (rel_drop > 0.05)
videos_with_big_drop = np.any(big_drop_mask, axis=1)
print(f"\nVideos with any drop >5% of prior day views: {videos_with_big_drop.sum()} / {len(df2)}")
print(f"Videos with any drop >20% of prior day views: {(np.any(rel_drop > 0.20, axis=1)).sum()}")

Top channel: UCckltLEhFLv8Xz_lQhYfwmg
Video count: 46485
Sample titles:
0     ජනපති අණින් නැවතූ සුළං බලාගාර ව්‍යාපෘතිය ගැන ඉ...
1     උතුම් නිකිණි පුන් පොහෝ දිනයේ මියුගුණාරාම රජමහා...
13    "පිටරටින් හාල් ගේන්නේ කොමිස් ගහන්න කිව්වා.. හැ...
16    හිරු සවස 6.55 ප්‍රධාන ප්‍රවෘත්ති විකාශය - Hiru...
20    නුවරඑළියේ අයාලේ යන පෝනියන්වෙන්දේසි කිරීමේ තීරණ...
23    ''33% විදුලි බිල අඩු කරනවා කිව්වා...එත් බලයට ආ...
24    හිරු මධ්‍යාහ්න 11.55 ප්‍රධාන ප්‍රවෘත්ති ප්‍රකා...
25    කුඹුරකට පැන්න වන අලි රංචුව - බැලු බැලු අත වන අ...
26    වනාතමුල්ලේ වෙඩි වරුසාව ගැන තවත් තොරතුරු රැසක් ...
27    ''පාර්ලිමේන්තුව කෝපි කඩේ වගේ කරන්න බැහැ!" - ''...

Category distribution:
category_id
25    46485
Name: count, dtype: int64

Avg duration: 0    PT48S
Name: video_duration, dtype: object
Publish frequency: 124.3 videos/day

Relative size of drops (as fraction of prior day's views), where decreases occur:
count    4.082600e+04
mean     1.370679e-02
std      6.572375e-02
min      6.092761e-07
25%      6.139364e-04
50%

In [33]:
import pandas as pd

our_channel_ids = set(videos_df['channel_id'].unique())
their_channel_ids = set(df2['channel_id'].unique())
new_channels = their_channel_ids - our_channel_ids

# build a summary per new channel to help you prioritize which to actually start tracking
# (1071 is a lot -- you may not want/need to poll all of them)
new_channel_summary = (
    df2[df2['channel_id'].isin(new_channels)]
    .groupby('channel_id')
    .agg(
        video_count=('id', 'count'),
        avg_day7_views=('day_7_views', 'mean'),
        median_day7_views=('day_7_views', 'median'),
        category_mode=('category_id', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        first_seen=('published_at', 'min'),
        last_seen=('published_at', 'max'),
    )
    .sort_values('video_count', ascending=False)
)

print(f"New channels available: {len(new_channel_summary)}")
print(new_channel_summary.head(20))

new_channel_summary.to_csv('new_channels_to_add.csv')
print("\nSaved new_channels_to_add.csv")

New channels available: 1071
                          video_count  avg_day7_views  median_day7_views  \
channel_id                                                                 
UCOtYyt7W5PmPnwQjWWF_Z-Q        13939    47202.982802             3540.5   
UCQTcNhAZidy1i9wwmdgf2Lw         7855    44861.491888             2664.0   
UCRDDfbYPHX_GUJ4lcQYTc8A         6964    49992.232360            12259.0   
UCNewfcqo9rMNNv8NzvHfQqQ         4760      365.941353               95.0   
UC2sO_F_Ky4LIAUtSJYXqBDA         2598     8490.146672              888.0   
UCYweMykEAI5aaRztqrJuuAg         2301      430.957061              293.0   
UCf16fS0s7Bha0efxVONTNdg         2136      305.798077               32.5   
UC0tV1b7IFEXFUUILF7mAcWw         2113       88.766164               26.0   
UCtmTjmQNsvvkCULd-z_1kow         2024      556.796859              324.5   
UCT83ymyAGm7Gnk_4ifxjxIA         1757    58319.318343             9617.0   
UC5va7JbC_k7ovI8P27FjNHQ         1458      570.173601      

In [34]:
import pandas as pd
import numpy as np

day_view_cols = [f"day_{i}_views" for i in range(1, 31)]
MAX_PER_CHANNEL = 50   # cap any single channel's contribution
H_DAYS = 5             # matches your H=120h choice

# --- 1. flag videos with a large real drop (exclude from labels, not from retrieval bank later) ---
vals = df2[day_view_cols].values.astype(float)
diffs = np.diff(vals, axis=1)
rel_drop = np.where(diffs < 0, -diffs / np.maximum(vals[:, :-1], 1), 0)
df2['has_big_drop'] = np.any(rel_drop > 0.20, axis=1)
print(f"Excluding {df2['has_big_drop'].sum()} videos with a >20% single-day drop")

clean2 = df2[~df2['has_big_drop']].copy()

# --- 2. fix small noise via cummax (monotonic enforcement) ---
vals_clean = clean2[day_view_cols].values.astype(float)
vals_fixed = np.fmax.accumulate(np.nan_to_num(vals_clean, nan=-np.inf), axis=1)
vals_fixed[np.isnan(vals_clean)] = np.nan
clean2[day_view_cols] = vals_fixed

# --- 3. filter to videos usable for the magnitude/horizon label ---
clean2 = clean2[clean2[f'day_{H_DAYS}_views'].notna() & clean2['day_1_views'].notna()]
print(f"Videos with day_1 and day_{H_DAYS} present: {len(clean2)}")

# --- 4. cap per-channel contribution to prevent domination ---
# sort so the most recent video per channel comes first, then take top N per channel
clean2_sorted = clean2.sort_values(['channel_id', 'published_at'], ascending=[True, False])
capped2 = clean2_sorted.groupby('channel_id').head(MAX_PER_CHANNEL).reset_index(drop=True)

print(f"After capping at {MAX_PER_CHANNEL}/channel: {len(capped2)} videos, "
      f"{capped2['channel_id'].nunique()} channels")

per_channel_after = capped2.groupby('channel_id').size().sort_values(ascending=False)
print(f"Top channel share after cap: {per_channel_after.iloc[0]} / {len(capped2)} "
      f"= {per_channel_after.iloc[0]/len(capped2):.1%}")

capped2.to_csv('dataset2_magnitude_ready.csv', index=False)
print("\nSaved dataset2_magnitude_ready.csv")

Excluding 561 videos with a >20% single-day drop
Videos with day_1 and day_5 present: 201903
After capping at 50/channel: 26169 videos, 1074 channels
Top channel share after cap: 50 / 26169 = 0.2%

Saved dataset2_magnitude_ready.csv


In [35]:
import pandas as pd

zero_day7 = capped2_h7[capped2_h7['day_7_views'] == 0]
print(f"Videos with day_7_views == 0: {len(zero_day7)}")

if len(zero_day7) > 0:
    print("\nSample:")
    print(zero_day7[['id', 'title', 'published_at', 'day_1_views', 'day_7_views', 'view_count']].head(10).to_string())

    # check if day_1_views is also 0 -- suggests these never had real view tracking
    print(f"\nOf these, day_1_views also 0: {(zero_day7['day_1_views'] == 0).sum()}")

    # check current view_count vs the daily columns -- big mismatch would suggest a scraping bug
    print("\nview_count vs day_7_views comparison:")
    print((zero_day7['view_count'] - zero_day7['day_7_views']).describe())

NameError: name 'capped2_h7' is not defined

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit

H_DAYS = 7
day_cols_H = [f"day_{i}_views" for i in range(1, H_DAYS + 1)]

# --- rebuild filters for H=7 (reuse clean2 from the previous cleaning step: drop big-drop rows + cummax fix) ---
clean2_h7 = clean2[clean2[f'day_{H_DAYS}_views'].notna() & clean2['day_1_views'].notna()].copy()
print(f"Videos with day_1..day_{H_DAYS} present: {len(clean2_h7)}")

clean2_h7['published_at'] = pd.to_datetime(clean2_h7['published_at'])

# recap at H=7 (previous cap used day_5 filter, redo cleanly)
clean2_h7_sorted = clean2_h7.sort_values(['channel_id', 'published_at'], ascending=[True, False])
capped2_h7 = clean2_h7_sorted.groupby('channel_id').head(50).reset_index(drop=True)
print(f"After capping at 50/channel: {len(capped2_h7)} videos, {capped2_h7['channel_id'].nunique()} channels")

# ---------------------------------------------------------------
# g(d): population maturation curve, daily, from the FULL (uncapped) corpus
# ---------------------------------------------------------------
g_d = {}
for d in range(1, H_DAYS + 1):
    ratio = clean2_h7[f'day_{d}_views'] / clean2_h7[f'day_{H_DAYS}_views']
    g_d[d] = ratio.median()
print("\nMaturation curve g(d):", g_d)

# ---------------------------------------------------------------
# S: channel anchor, computed per video from that channel's PRIOR videos
# using the full (uncapped) history, day_H views directly (no rescale needed
# since day_H is always present in clean2_h7 by construction)
# ---------------------------------------------------------------
clean2_h7 = clean2_h7.sort_values('published_at').reset_index(drop=True)

def compute_S_for_channel(group, min_prior=5):
    group = group.sort_values('published_at')
    views_h = group[f'day_{H_DAYS}_views'].values
    dates = group['published_at'].values
    S_vals = np.full(len(group), np.nan)
    for i in range(len(group)):
        prior_views = views_h[:i]  # strictly before this video
        if len(prior_views) >= min_prior:
            S_vals[i] = np.median(prior_views[-30:])  # last 30 prior, median
    group = group.copy()
    group['S'] = S_vals
    return group

print("\nComputing S per video (this may take a moment)...")
clean2_h7 = clean2_h7.groupby('channel_id', group_keys=False).apply(compute_S_for_channel)

# merge S back onto the capped training set
capped2_h7 = capped2_h7.merge(
    clean2_h7[['id', 'S']], on='id', how='left'
)
capped2_h7 = capped2_h7[capped2_h7['S'].notna() & (capped2_h7['S'] > 0)].copy()

# drop videos with zero views at horizon -- log(m) undefined, and these likely
# indicate either genuinely negligible reach or a data issue (checked separately)
n_before = len(capped2_h7)
capped2_h7 = capped2_h7[capped2_h7[f'day_{H_DAYS}_views'] > 0].copy()
print(f"\nDropped {n_before - len(capped2_h7)} videos with day_{H_DAYS}_views == 0")
print(f"Training videos with valid S and nonzero views: {len(capped2_h7)}")

# ---------------------------------------------------------------
# m: magnitude label
# ---------------------------------------------------------------
capped2_h7['m'] = capped2_h7[f'day_{H_DAYS}_views'] / capped2_h7['S']
capped2_h7['log_m'] = np.log(capped2_h7['m'])

print("\nlog(m) distribution:")
print(capped2_h7['log_m'].describe())

# clip extremes (0.5th / 99.5th percentile) to stop outliers dominating the loss
lo, hi = capped2_h7['log_m'].quantile([0.005, 0.995])
capped2_h7['log_m_clipped'] = capped2_h7['log_m'].clip(lo, hi)
print(f"\nClipped log(m) range: [{lo:.2f}, {hi:.2f}]")

# ---------------------------------------------------------------
# F(t): shape parameters, single-component saturating curve
#   F(t) = 1 - (1 + t/c)^(-theta)
# fit per video via least squares on its own daily points
# ---------------------------------------------------------------
def shape_func(t, c, theta):
    return 1 - (1 + t / c) ** (-theta)

def fit_shape(row):
    if row[f'day_{H_DAYS}_views'] <= 0:
        return pd.Series({'c': np.nan, 'theta': np.nan, 'shape_r2': np.nan})
    t = np.arange(1, H_DAYS + 1)
    y = row[day_cols_H].values.astype(float) / row[f'day_{H_DAYS}_views']
    y = np.clip(y, 1e-6, 1.0)
    try:
        popt, _ = curve_fit(shape_func, t, y, p0=[2.0, 1.0], maxfev=2000,
                             bounds=([0.01, 0.01], [100, 20]))
        resid = y - shape_func(t, *popt)
        r2 = 1 - np.sum(resid**2) / np.sum((y - y.mean())**2)
        return pd.Series({'c': popt[0], 'theta': popt[1], 'shape_r2': r2})
    except Exception:
        return pd.Series({'c': np.nan, 'theta': np.nan, 'shape_r2': np.nan})

print("\nFitting shape curve per video (this may take a few minutes for ~25k rows)...")
shape_params = capped2_h7.apply(fit_shape, axis=1)
capped2_h7 = pd.concat([capped2_h7, shape_params], axis=1)

print("\nShape fit quality (R^2) across videos:")
print(capped2_h7['shape_r2'].describe())
print(f"\nVideos with R^2 >= 0.95: {(capped2_h7['shape_r2'] >= 0.95).sum()} / {len(capped2_h7)}")
print(f"Videos with R^2 < 0.8:   {(capped2_h7['shape_r2'] < 0.8).sum()}")

capped2_h7.to_csv('training_corpus_h7.csv', index=False)
print("\nSaved training_corpus_h7.csv")

Videos with day_1..day_7 present: 200562
After capping at 50/channel: 26059 videos, 1072 channels

Maturation curve g(d): {1: np.float64(0.8104868913857678), 2: np.float64(0.9330357142857143), 3: np.float64(0.9719774664162936), 4: np.float64(0.9874834060307226), 5: np.float64(0.9940828402366864), 6: np.float64(0.998015873015873), 7: np.float64(1.0)}

Computing S per video (this may take a moment)...

Dropped 274 videos with day_7_views == 0
Training videos with valid S and nonzero views: 22628

log(m) distribution:
count    22628.000000
mean        -0.060652
std          1.639120
min         -7.801299
25%         -0.871839
50%         -0.036996
75%          0.751249
max         10.632706
Name: log_m, dtype: float64

Clipped log(m) range: [-5.19, 5.11]

Fitting shape curve per video (this may take a few minutes for ~25k rows)...


C:\Users\ASUS\AppData\Local\Temp\ipykernel_20804\1693977597.py:96: RuntimeWarning: divide by zero encountered in scalar divide
  r2 = 1 - np.sum(resid**2) / np.sum((y - y.mean())**2)



Shape fit quality (R^2) across videos:
count    2.262800e+04
mean             -inf
std               NaN
min              -inf
25%      8.535769e-01
50%      9.395056e-01
75%      9.771488e-01
max      9.999819e-01
Name: shape_r2, dtype: float64

Videos with R^2 >= 0.95: 10088 / 22628
Videos with R^2 < 0.8:   3849


C:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)



Saved training_corpus_h7.csv


In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit

day_cols_H = [f"day_{i}_views" for i in range(1, 8)]

# --- 1. what do the worst fits look like? ---
worst = training_corpus_h7.nsmallest(10, 'shape_r2') if 'training_corpus_h7' in dir() else capped2_h7.nsmallest(10, 'shape_r2')
print("Worst-fit videos, actual daily ratios (day_t / day_7):")
for _, row in worst.iterrows():
    ratios = [row[c] / row['day_7_views'] for c in day_cols_H]
    print(f"  R2={row['shape_r2']:.3f}  c={row['c']:.2f} theta={row['theta']:.2f}  ratios={[f'{r:.3f}' for r in ratios]}")

# --- 2. is low R2 correlated with anything (category, duration, m)? ---
low_r2 = capped2_h7[capped2_h7['shape_r2'] < 0.8]
high_r2 = capped2_h7[capped2_h7['shape_r2'] >= 0.95]
print(f"\nLow-R2 group: median log_m={low_r2['log_m'].median():.2f}, "
      f"category mode={low_r2['category_id'].mode().iloc[0] if len(low_r2) else 'NA'}")
print(f"High-R2 group: median log_m={high_r2['log_m'].median():.2f}, "
      f"category mode={high_r2['category_id'].mode().iloc[0] if len(high_r2) else 'NA'}")

# --- 3. try an alternative 2-parameter form: logistic saturation ---
def shape_func_logistic(t, k, t0):
    return 1 / (1 + np.exp(-k * (t - t0)))

def fit_both_forms(row):
    t = np.arange(1, 8)
    y = row[day_cols_H].values.astype(float) / row['day_7_views']
    y = np.clip(y, 1e-6, 1.0)
    results = {}

    # power-law form (current)
    try:
        popt, _ = curve_fit(lambda t, c, th: 1 - (1 + t/c)**(-th), t, y,
                             p0=[2.0, 1.0], maxfev=2000, bounds=([0.01,0.01],[100,20]))
        resid = y - (1 - (1 + t/popt[0])**(-popt[1]))
        var_y = np.var(y)
        r2_power = (1.0 if np.sum(resid**2) < 1e-8 else np.nan) if var_y < 1e-10 \
                   else 1 - np.sum(resid**2)/np.sum((y-y.mean())**2)
    except Exception:
        r2_power = np.nan

    # logistic form
    try:
        # normalize logistic to hit y(7)=1 by construction issue -- rescale
        popt2, _ = curve_fit(shape_func_logistic, t, y, p0=[1.0, 0.5],
                              maxfev=2000, bounds=([0.01, -5], [10, 7]))
        pred = shape_func_logistic(t, *popt2)
        pred = pred / pred[-1]  # force F(7)=1
        resid2 = y - pred
        var_y = np.var(y)
        r2_logistic = (1.0 if np.sum(resid2**2) < 1e-8 else np.nan) if var_y < 1e-10 \
                      else 1 - np.sum(resid2**2)/np.sum((y-y.mean())**2)
    except Exception:
        r2_logistic = np.nan

    return pd.Series({'r2_power': r2_power, 'r2_logistic': r2_logistic})

print("\nComparing forms on a 2000-row sample...")
sample = capped2_h7.sample(min(2000, len(capped2_h7)), random_state=0)
comparison = sample.apply(fit_both_forms, axis=1)
print("\nPower-law R2:")
print(comparison['r2_power'].describe())
print("\nLogistic R2:")
print(comparison['r2_logistic'].describe())

Worst-fit videos, actual daily ratios (day_t / day_7):
  R2=-inf  c=0.69 theta=12.38  ratios=['1.000', '1.000', '1.000', '1.000', '1.000', '1.000', '1.000']
  R2=-inf  c=0.69 theta=12.38  ratios=['1.000', '1.000', '1.000', '1.000', '1.000', '1.000', '1.000']
  R2=-inf  c=0.69 theta=12.38  ratios=['1.000', '1.000', '1.000', '1.000', '1.000', '1.000', '1.000']
  R2=-inf  c=0.69 theta=12.38  ratios=['1.000', '1.000', '1.000', '1.000', '1.000', '1.000', '1.000']
  R2=-inf  c=0.69 theta=12.38  ratios=['1.000', '1.000', '1.000', '1.000', '1.000', '1.000', '1.000']
  R2=-inf  c=0.69 theta=12.38  ratios=['1.000', '1.000', '1.000', '1.000', '1.000', '1.000', '1.000']
  R2=-inf  c=0.69 theta=12.38  ratios=['1.000', '1.000', '1.000', '1.000', '1.000', '1.000', '1.000']
  R2=-inf  c=0.69 theta=12.38  ratios=['1.000', '1.000', '1.000', '1.000', '1.000', '1.000', '1.000']
  R2=-inf  c=0.69 theta=12.38  ratios=['1.000', '1.000', '1.000', '1.000', '1.000', '1.000', '1.000']
  R2=-inf  c=0.69 theta=12.

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit

day_cols_H = [f"day_{i}_views" for i in range(1, 8)]

def power_form(t, c, theta):
    return 1 - (1 + t / c) ** (-theta)

def logistic_form(t, k, t0):
    raw = 1 / (1 + np.exp(-k * (t - t0)))
    return raw

def safe_r2(y, pred):
    resid = y - pred
    var_y = np.var(y)
    if var_y < 1e-10:
        return 1.0 if np.sum(resid**2) < 1e-8 else np.nan
    return 1 - np.sum(resid**2) / np.sum((y - y.mean())**2)

def fit_hybrid(row):
    if row['day_7_views'] <= 0:
        return pd.Series({'c': np.nan, 'theta': np.nan, 'k': np.nan, 't0': np.nan,
                           'shape_r2': np.nan, 'shape_form': None})
    t = np.arange(1, 8)
    y = row[day_cols_H].values.astype(float) / row['day_7_views']
    y = np.clip(y, 1e-6, 1.0)

    # power-law attempt
    r2_power, params_power = np.nan, (np.nan, np.nan)
    try:
        popt, _ = curve_fit(power_form, t, y, p0=[2.0, 1.0], maxfev=2000,
                             bounds=([0.01, 0.01], [100, 20]))
        r2_power = safe_r2(y, power_form(t, *popt))
        params_power = tuple(popt)
    except Exception:
        pass

    # logistic attempt
    r2_logistic, params_logistic = np.nan, (np.nan, np.nan)
    try:
        popt2, _ = curve_fit(logistic_form, t, y, p0=[1.0, 0.5],
                              maxfev=2000, bounds=([0.01, -5], [10, 7]))
        pred = logistic_form(t, *popt2)
        pred = pred / pred[-1]  # force F(7)=1 exactly
        r2_logistic = safe_r2(y, pred)
        params_logistic = tuple(popt2)
    except Exception:
        pass

    # pick the winner
    r2_power_c = -np.inf if np.isnan(r2_power) else r2_power
    r2_logistic_c = -np.inf if np.isnan(r2_logistic) else r2_logistic

    if r2_power_c >= r2_logistic_c and not np.isnan(r2_power):
        return pd.Series({'c': params_power[0], 'theta': params_power[1],
                           'k': np.nan, 't0': np.nan,
                           'shape_r2': r2_power, 'shape_form': 'power'})
    elif not np.isnan(r2_logistic):
        return pd.Series({'c': np.nan, 'theta': np.nan,
                           'k': params_logistic[0], 't0': params_logistic[1],
                           'shape_r2': r2_logistic, 'shape_form': 'logistic'})
    else:
        return pd.Series({'c': np.nan, 'theta': np.nan, 'k': np.nan, 't0': np.nan,
                           'shape_r2': np.nan, 'shape_form': None})

print("Fitting hybrid shape (power + logistic, keep best) for all training videos...")
shape_results = capped2_h7.apply(fit_hybrid, axis=1)
capped2_h7_final = pd.concat([capped2_h7.drop(columns=['c','theta','shape_r2'], errors='ignore'),
                               shape_results], axis=1)

print("\nShape form chosen:")
print(capped2_h7_final['shape_form'].value_counts())

print("\nFinal R2 distribution:")
print(capped2_h7_final['shape_r2'].describe())
print(f"\nR2 >= 0.95: {(capped2_h7_final['shape_r2'] >= 0.95).sum()} / {len(capped2_h7_final)}")
print(f"R2 < 0.8:   {(capped2_h7_final['shape_r2'] < 0.8).sum()}")

capped2_h7_final.to_csv('training_corpus_h7_final.csv', index=False)
print("\nSaved training_corpus_h7_final.csv")

Fitting hybrid shape (power + logistic, keep best) for all training videos...

Shape form chosen:
shape_form
logistic    15308
power        7320
Name: count, dtype: int64

Final R2 distribution:
count    22628.000000
mean         0.943771
std          0.135786
min         -2.804209
25%          0.947892
50%          0.981346
75%          0.994233
max          1.000000
Name: shape_r2, dtype: float64

R2 >= 0.95: 16777 / 22628
R2 < 0.8:   1425

Saved training_corpus_h7_final.csv


In [ ]:
capped2_h7_final['shape_usable'] = capped2_h7_final['shape_r2'] >= 0.8
print(f"Usable for shape loss: {capped2_h7_final['shape_usable'].sum()}")
print(f"Magnitude-only (shape excluded): {(~capped2_h7_final['shape_usable']).sum()}")

capped2_h7_final.to_csv('training_corpus_h7_final.csv', index=False)

Usable for shape loss: 21203
Magnitude-only (shape excluded): 1425


In [ ]:
import pandas as pd

usable = capped2_h7_final[capped2_h7_final['shape_usable']]

# for power-law videos: does theta or c correlate with log_m?
power_vids = usable[usable['shape_form'] == 'power']
print("Power-law videos: correlation with log_m")
print(f"  theta vs log_m: {power_vids['theta'].corr(power_vids['log_m']):.3f}")
print(f"  c vs log_m:     {power_vids['c'].corr(power_vids['log_m']):.3f}")

# for logistic videos: does k or t0 correlate with log_m?
logistic_vids = usable[usable['shape_form'] == 'logistic']
print("\nLogistic videos: correlation with log_m")
print(f"  k vs log_m:  {logistic_vids['k'].corr(logistic_vids['log_m']):.3f}")
print(f"  t0 vs log_m: {logistic_vids['t0'].corr(logistic_vids['log_m']):.3f}")

# does the CHOICE of form itself correlate with magnitude?
print("\nMedian log_m by shape form:")
print(usable.groupby('shape_form')['log_m'].median())

Power-law videos: correlation with log_m
  theta vs log_m: -0.006
  c vs log_m:     0.041

Logistic videos: correlation with log_m
  k vs log_m:  -0.036
  t0 vs log_m: 0.002

Median log_m by shape form:
shape_form
logistic    0.021053
power      -0.115932
Name: log_m, dtype: float64


In [36]:
pip install sentence-transformers pillow requests torch tqdm --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [37]:
"""
Run locally. GPU recommended (falls back to CPU, just slower -- expect
roughly 20-40 min on GPU for ~23k videos, several hours on CPU).
"""

import pandas as pd
import numpy as np
import requests
from PIL import Image
from io import BytesIO
from sentence_transformers import SentenceTransformer
import torch
import re
from tqdm import tqdm

tqdm.pandas()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

df = pd.read_csv('training_corpus_h7_final.csv')
print(f"Videos to process: {len(df)}")

# ---------------------------------------------------------------
# 1. Load models
# ---------------------------------------------------------------
text_model = SentenceTransformer('clip-ViT-B-32-multilingual-v1', device=device)
image_model = SentenceTransformer('clip-ViT-B-32', device=device)

# ---------------------------------------------------------------
# 2. Text embeddings: title + tags (description often noisy/boilerplate,
#    weight title higher by putting it first and repeating once)
# ---------------------------------------------------------------
def build_text(row):
    title = str(row['title']) if pd.notna(row['title']) else ""
    tags = row['tags']
    try:
        import json
        tag_list = json.loads(tags) if isinstance(tags, str) else []
        tag_str = " ".join(tag_list[:10])  # cap to avoid tag-spam dominating
    except Exception:
        tag_str = ""
    return f"{title}. {title}. {tag_str}"  # title counted twice for emphasis

df['text_input'] = df.apply(build_text, axis=1)

print("Encoding text...")
text_embeddings = text_model.encode(
    df['text_input'].tolist(), batch_size=64, show_progress_bar=True,
    convert_to_numpy=True
)
np.save('text_embeddings.npy', text_embeddings)
print(f"Text embeddings shape: {text_embeddings.shape}")

# ---------------------------------------------------------------
# 3. Thumbnail embeddings: download + encode in batches
# ---------------------------------------------------------------
def download_image(url, timeout=5):
    try:
        resp = requests.get(url, timeout=timeout)
        img = Image.open(BytesIO(resp.content)).convert('RGB')
        return img
    except Exception:
        return None

print("Downloading and encoding thumbnails...")
BATCH_SIZE = 64
image_embeddings = np.zeros((len(df), 512), dtype=np.float32)  # clip-ViT-B-32 dim
failed_indices = []

urls = df['thumbnail_medium'].fillna(df['thumbnail_default']).tolist()

for start in tqdm(range(0, len(df), BATCH_SIZE)):
    end = min(start + BATCH_SIZE, len(df))
    batch_urls = urls[start:end]
    batch_images = []
    batch_valid_idx = []

    for i, url in enumerate(batch_urls):
        img = download_image(url)
        if img is not None:
            batch_images.append(img)
            batch_valid_idx.append(start + i)
        else:
            failed_indices.append(start + i)

    if batch_images:
        embs = image_model.encode(batch_images, convert_to_numpy=True)
        for idx, emb in zip(batch_valid_idx, embs):
            image_embeddings[idx] = emb

print(f"Failed thumbnail downloads: {len(failed_indices)} / {len(df)}")
np.save('image_embeddings.npy', image_embeddings)

# ---------------------------------------------------------------
# 4. Thumbnail-title alignment score (cosine similarity, same embedding space)
# ---------------------------------------------------------------
def cosine_sim_batch(a, b):
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
    return np.sum(a_norm * b_norm, axis=1)

df['thumbnail_title_alignment'] = cosine_sim_batch(image_embeddings, text_embeddings)
# zero out alignment for failed thumbnail downloads -- it's meaningless there
df.loc[failed_indices, 'thumbnail_title_alignment'] = np.nan

print("\nAlignment score distribution:")
print(df['thumbnail_title_alignment'].describe())

# ---------------------------------------------------------------
# 5. Tabular features
# ---------------------------------------------------------------
def parse_duration(iso):
    if pd.isna(iso):
        return np.nan
    m = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", str(iso))
    if not m:
        return np.nan
    h, mi, s = (int(x) if x else 0 for x in m.groups())
    return h * 3600 + mi * 60 + s

df['duration_s'] = df['video_duration'].apply(parse_duration)
df['title_length'] = df['title'].fillna('').str.len()
df['description_length'] = df['description'].fillna('').str.len()

def count_tags(t):
    try:
        import json
        return len(json.loads(t)) if isinstance(t, str) else 0
    except Exception:
        return 0

df['tag_count'] = df['tags'].apply(count_tags)

df['published_at'] = pd.to_datetime(df['published_at'])
df['publish_hour_sin'] = np.sin(2 * np.pi * df['published_at'].dt.hour / 24)
df['publish_hour_cos'] = np.cos(2 * np.pi * df['published_at'].dt.hour / 24)
df['publish_dow_sin'] = np.sin(2 * np.pi * df['published_at'].dt.dayofweek / 7)
df['publish_dow_cos'] = np.cos(2 * np.pi * df['published_at'].dt.dayofweek / 7)

# ---------------------------------------------------------------
# 6. Save everything
# ---------------------------------------------------------------
tabular_cols = ['id', 'video_id' if 'video_id' in df.columns else 'id', 'channel_id',
                 'category_id', 'duration_s', 'title_length', 'description_length',
                 'tag_count', 'publish_hour_sin', 'publish_hour_cos',
                 'publish_dow_sin', 'publish_dow_cos', 'thumbnail_title_alignment',
                 'log_m', 'log_m_clipped', 'shape_form', 'shape_r2', 'shape_usable',
                 'c', 'theta', 'k', 't0']
tabular_cols = [c for c in dict.fromkeys(tabular_cols) if c in df.columns]

df[tabular_cols].to_parquet('features_tabular.parquet', index=False)
print(f"\nSaved features_tabular.parquet: {df[tabular_cols].shape}")
print("Saved text_embeddings.npy and image_embeddings.npy (row-aligned with the parquet)")

c:\Users\ASUS\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Videos to process: 22628


c:\Users\ASUS\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--sentence-transformers--clip-ViT-B-32-multilingual-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3441.88it/s]
c:\Users\ASUS\miniconda3

Encoding text...


Batches: 100%|██████████| 354/354 [24:44<00:00,  4.19s/it]


Text embeddings shape: (22628, 512)


  0%|          | 0/354 [00:42<?, ?it/s]


KeyboardInterrupt: 

In [40]:
import requests
from PIL import Image
from io import BytesIO

def download_image(url, timeout=5):
    try:
        resp = requests.get(url, timeout=timeout)
        img = Image.open(BytesIO(resp.content)).convert('RGB')
        return img
    except Exception:
        return None

print("Downloading and encoding thumbnails...")
from concurrent.futures import ThreadPoolExecutor
import requests as requests_lib

BATCH_SIZE = 64
CHECKPOINT_EVERY = 20  # batches
image_embeddings = np.zeros((len(df), 512), dtype=np.float32)
failed_indices = []

urls = df['thumbnail_medium'].fillna(df['thumbnail_default']).tolist()

# one shared session with connection pooling + a real User-Agent
session = requests_lib.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0'})
adapter = requests_lib.adapters.HTTPAdapter(pool_connections=32, pool_maxsize=32)
session.mount('https://', adapter)
session.mount('http://', adapter)

def download_one(args):
    idx, url = args
    try:
        resp = session.get(url, timeout=4)
        img = Image.open(BytesIO(resp.content)).convert('RGB')
        return idx, img
    except Exception:
        return idx, None

# one thread pool for the whole run, not recreated per batch
executor = ThreadPoolExecutor(max_workers=32)

for b, start in enumerate(tqdm(range(0, len(df), BATCH_SIZE), total=(len(df)//BATCH_SIZE)+1)):
    end = min(start + BATCH_SIZE, len(df))
    batch_items = list(enumerate(urls[start:end], start=start))
    batch_images = []
    batch_valid_idx = []

    for idx, img in executor.map(download_one, batch_items):
        if img is not None:
            batch_images.append(img)
            batch_valid_idx.append(idx)
        else:
            failed_indices.append(idx)

    if batch_images:
        embs = image_model.encode(batch_images, convert_to_numpy=True)
        for idx, emb in zip(batch_valid_idx, embs):
            image_embeddings[idx] = emb

    if b % CHECKPOINT_EVERY == 0:
        np.save('image_embeddings_checkpoint.npy', image_embeddings)

executor.shutdown()
print(f"Failed thumbnail downloads: {len(failed_indices)} / {len(df)}")
np.save('image_embeddings.npy', image_embeddings)

  0%|          | 0/354 [00:00<?, ?it/s]

100%|██████████| 354/354 [1:20:05<00:00, 13.57s/it]

Failed thumbnail downloads: 2818 / 22628


In [41]:
import pandas as pd
import numpy as np

# --- 1. check if failures cluster by video age (older thumbnails more likely dead) ---
failed_df = df.iloc[failed_indices]
print(f"Failed downloads: {len(failed_df)} / {len(df)}")
print("\nPublish date range of failed videos:")
print(failed_df['published_at'].describe())
print("\nPublish date range of ALL videos (for comparison):")
print(df['published_at'].describe())

# check a few actual URLs and what error they threw
print("\nSample failed URLs:")
sample_fail_urls = df.iloc[failed_indices[:5]]['thumbnail_medium'].fillna(
    df.iloc[failed_indices[:5]]['thumbnail_default']
)
for url in sample_fail_urls:
    try:
        r = session.get(url, timeout=5)
        print(f"{url} -> status {r.status_code}, {len(r.content)} bytes")
    except Exception as e:
        print(f"{url} -> ERROR: {e}")

# --- 2. retry failed downloads once (recovers transient timeouts) ---
print(f"\nRetrying {len(failed_indices)} failed downloads...")
retry_urls = [(idx, urls[idx]) for idx in failed_indices]
still_failed = []
recovered = 0

BATCH = 64
for i in tqdm(range(0, len(retry_urls), BATCH)):
    batch = retry_urls[i:i+BATCH]
    batch_images = []
    batch_valid_idx = []

    for idx, img in executor.map(download_one, batch):
        if img is not None:
            batch_images.append(img)
            batch_valid_idx.append(idx)
        else:
            still_failed.append(idx)

    if batch_images:
        embs = image_model.encode(batch_images, convert_to_numpy=True)
        for idx, emb in zip(batch_valid_idx, embs):
            image_embeddings[idx] = emb
        recovered += len(batch_valid_idx)

print(f"\nRecovered on retry: {recovered}")
print(f"Still failed after retry: {len(still_failed)} / {len(df)}")

np.save('image_embeddings.npy', image_embeddings)

# --- 3. flag remaining failures so the model can learn to ignore them, not trust a zero vector blindly ---
df['has_thumbnail'] = True
df.loc[still_failed, 'has_thumbnail'] = False
print(f"\nFinal: {df['has_thumbnail'].sum()} with thumbnails, {(~df['has_thumbnail']).sum()} without")

Failed downloads: 2818 / 22628

Publish date range of failed videos:
count                    2818
unique                   2818
top       2026-08-06 20:30:39
freq                        1
Name: published_at, dtype: object

Publish date range of ALL videos (for comparison):
count                   22628
unique                  22528
top       2026-08-08 16:30:06
freq                        3
Name: published_at, dtype: object

Sample failed URLs:
https://i.ytimg.com/vi/uoCi2dSKOe0/mqdefault.jpg -> status 200, 20609 bytes
https://i.ytimg.com/vi/Edw1LVwyCV4/mqdefault.jpg -> status 200, 20973 bytes
https://i.ytimg.com/vi/g5WxLQYkd4g/mqdefault.jpg -> status 200, 21093 bytes
https://i.ytimg.com/vi/3btg2VvqhD0/mqdefault.jpg -> status 200, 5987 bytes
https://i.ytimg.com/vi/HMRnmQCH_MM/mqdefault.jpg -> status 200, 5362 bytes

Retrying 2818 failed downloads...


  0%|          | 0/45 [00:00<?, ?it/s]


RuntimeError: cannot schedule new futures after shutdown

In [7]:
import pandas as pd
import numpy as np
import re
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit

# ---------------------------------------------------------------
# 0. Tabular features (computed here directly in case the earlier
#    extraction cell wasn't run in this session)
# ---------------------------------------------------------------
def parse_duration(iso):
    if pd.isna(iso):
        return np.nan
    m = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", str(iso))
    if not m:
        return np.nan
    h, mi, s = (int(x) if x else 0 for x in m.groups())
    return h * 3600 + mi * 60 + s

if 'duration_s' not in df.columns:
    df['duration_s'] = df['video_duration'].apply(parse_duration)

if 'title_length' not in df.columns:
    df['title_length'] = df['title'].fillna('').str.len()

if 'description_length' not in df.columns:
    df['description_length'] = df['description'].fillna('').str.len()

if 'tag_count' not in df.columns:
    def count_tags(t):
        try:
            import json
            return len(json.loads(t)) if isinstance(t, str) else 0
        except Exception:
            return 0
    df['tag_count'] = df['tags'].apply(count_tags)

if 'publish_hour_sin' not in df.columns:
    df['published_at'] = pd.to_datetime(df['published_at'])
    df['publish_hour_sin'] = np.sin(2 * np.pi * df['published_at'].dt.hour / 24)
    df['publish_hour_cos'] = np.cos(2 * np.pi * df['published_at'].dt.hour / 24)
    df['publish_dow_sin'] = np.sin(2 * np.pi * df['published_at'].dt.dayofweek / 7)
    df['publish_dow_cos'] = np.cos(2 * np.pi * df['published_at'].dt.dayofweek / 7)

print("Tabular features ready:", df[['duration_s','title_length','description_length',
                                       'tag_count','publish_hour_sin']].isna().sum().to_dict())

# ---------------------------------------------------------------
# 1. Finalize thumbnail embeddings: flag the still-failed ones,
#    fill with the MEAN embedding (better than zero -- zero would
#    look like a specific point in embedding space, mean is neutral)
# ---------------------------------------------------------------
had_thumbnail = np.any(image_embeddings != 0, axis=1)
df['has_thumbnail'] = had_thumbnail
print(f"Videos with thumbnail: {had_thumbnail.sum()} / {len(df)}")

mean_img_emb = image_embeddings[had_thumbnail].mean(axis=0)
image_embeddings[~had_thumbnail] = mean_img_emb

np.save('image_embeddings_final.npy', image_embeddings)

# ---------------------------------------------------------------
# 2. Recompute alignment score now that missing ones are filled
#    (mark alignment as 0 / neutral for videos without a real thumbnail)
# ---------------------------------------------------------------
def cosine_sim_batch(a, b):
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
    return np.sum(a_norm * b_norm, axis=1)

df['thumbnail_title_alignment'] = cosine_sim_batch(image_embeddings, text_embeddings)
df.loc[~had_thumbnail, 'thumbnail_title_alignment'] = 0.0

# ---------------------------------------------------------------
# 3. Compress embeddings via PCA for fast training (raw 512+512 dims
#    would work but PCA speeds up the quick baseline substantially)
# ---------------------------------------------------------------
N_COMPONENTS = 32
text_pca = PCA(n_components=N_COMPONENTS, random_state=0).fit_transform(text_embeddings)
img_pca = PCA(n_components=N_COMPONENTS, random_state=0).fit_transform(image_embeddings)

text_cols = [f'text_pc{i}' for i in range(N_COMPONENTS)]
img_cols = [f'img_pc{i}' for i in range(N_COMPONENTS)]
df[text_cols] = text_pca
df[img_cols] = img_pca

# ---------------------------------------------------------------
# 4. Build the final feature matrix
# ---------------------------------------------------------------
feature_cols = (
    ['duration_s', 'title_length', 'description_length', 'tag_count',
     'publish_hour_sin', 'publish_hour_cos', 'publish_dow_sin', 'publish_dow_cos',
     'thumbnail_title_alignment', 'has_thumbnail']
    + text_cols + img_cols
)
# category as one-hot (fine for a quick GBM baseline)
df = df.loc[:, ~df.columns.str.startswith('cat_')]
cat_dummies = pd.get_dummies(df['category_id'], prefix='cat')
df = pd.concat([df, cat_dummies], axis=1)
df = pd.concat([df, cat_dummies], axis=1)
feature_cols += list(cat_dummies.columns)

X = df[feature_cols].fillna(0)
X = X.loc[:, ~X.columns.duplicated()]
y = df['log_m_clipped']
groups = df['channel_id']

print(f"\nFeature matrix: {X.shape}")

# ---------------------------------------------------------------
# 5. Channel-disjoint split
# ---------------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=0)
train_idx, temp_idx = next(gss.split(X, y, groups))

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_temp, y_temp, groups_temp = X.iloc[temp_idx], y.iloc[temp_idx], groups.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=0)
val_idx, test_idx = next(gss2.split(X_temp, y_temp, groups_temp))
X_val, y_val = X_temp.iloc[val_idx], y_temp.iloc[val_idx]
X_test, y_test = X_temp.iloc[test_idx], y_temp.iloc[test_idx]

print(f"Train: {len(X_train)} ({groups.iloc[train_idx].nunique()} channels)")
print(f"Val:   {len(X_val)} ({groups_temp.iloc[val_idx].nunique()} channels)")
print(f"Test:  {len(X_test)} ({groups_temp.iloc[test_idx].nunique()} channels)")

# ---------------------------------------------------------------
# 6. Baseline: predict m=1 (log_m=0) for everyone
# ---------------------------------------------------------------
from sklearn.metrics import mean_absolute_error, r2_score

baseline_pred = np.zeros(len(y_val))
baseline_mae = mean_absolute_error(y_val, baseline_pred)
baseline_r2 = r2_score(y_val, baseline_pred)
print(f"\nBASELINE (m=1 always): MAE={baseline_mae:.3f}, R2={baseline_r2:.3f}")

# ---------------------------------------------------------------
# 7. Quick model: CatBoost (fast, handles mixed features well, no GPU needed)
# ---------------------------------------------------------------
try:
    from catboost import CatBoostRegressor
    model = CatBoostRegressor(iterations=500, depth=6, learning_rate=0.05,
                               loss_function='MAE', verbose=100, random_state=0)
except ImportError:
    print("CatBoost not installed -- run: pip install catboost --break-system-packages")
    raise

model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

val_pred = model.predict(X_val)
val_mae = mean_absolute_error(y_val, val_pred)
val_r2 = r2_score(y_val, val_pred)
print(f"\nCATBOOST: MAE={val_mae:.3f}, R2={val_r2:.3f}")
print(f"Improvement over baseline: {(1 - val_mae/baseline_mae)*100:.1f}% lower MAE")

# feature importance -- quick sanity check on what's driving predictions
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nTop 15 features:")
print(importances.head(15))

Tabular features ready: {'duration_s': 302, 'title_length': 0, 'description_length': 0, 'tag_count': 0, 'publish_hour_sin': 0}


NameError: name 'image_embeddings' is not defined

In [51]:
import pandas as pd
import numpy as np

# NOTE: ideally these stats come from each channel's PRIOR videos only (like S was
# computed earlier), using the full uncapped history. clean2_h7 (that full history)
# isn't in memory in this session, so as a quick check we compute these stats from
# df itself (the capped 22,628-video training set). This means each video's own
# value is included in its channel's stats -- a small amount of leakage, acceptable
# for this quick diagnostic, but NOT how the real model should do it (should reuse
# the point-in-time logic from compute_S_for_channel instead, before finalizing).

channel_stats = df.groupby('channel_id').agg(
    channel_video_count=('id', 'count'),
    channel_median_views=('day_7_views', 'median'),
    channel_view_std=('day_7_views', 'std'),
).reset_index()

channel_stats['channel_log_volatility'] = (
    df.groupby('channel_id')['day_7_views']
    .apply(lambda x: np.std(np.log(x.clip(lower=1))))
    .values
)

# category diversity: how many distinct categories does this channel post in?
channel_stats['channel_category_diversity'] = (
    df.groupby('channel_id')['category_id'].nunique().values
)

print(f"Channel stats computed for {len(channel_stats)} channels")
print(channel_stats.describe())

# --- merge onto the training set ---
df_v2 = df.merge(channel_stats, on='channel_id', how='left')

channel_feature_cols = ['channel_video_count', 'channel_median_views', 'channel_view_std',
                         'channel_log_volatility', 'channel_category_diversity']

X_v2 = df_v2[feature_cols + channel_feature_cols].fillna(0)
X_v2 = X_v2.loc[:, ~X_v2.columns.duplicated()]
y_v2 = df_v2['log_m_clipped']
groups_v2 = df_v2['channel_id']

# same channel-disjoint split
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=0)
train_idx, temp_idx = next(gss.split(X_v2, y_v2, groups_v2))
X_train2, y_train2 = X_v2.iloc[train_idx], y_v2.iloc[train_idx]
X_temp2, y_temp2, groups_temp2 = X_v2.iloc[temp_idx], y_v2.iloc[temp_idx], groups_v2.iloc[temp_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=0)
val_idx2, test_idx2 = next(gss2.split(X_temp2, y_temp2, groups_temp2))
X_val2, y_val2 = X_temp2.iloc[val_idx2], y_temp2.iloc[val_idx2]

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, r2_score

model2 = CatBoostRegressor(iterations=500, depth=6, learning_rate=0.05,
                            loss_function='MAE', verbose=100, random_state=0)
model2.fit(X_train2, y_train2, eval_set=(X_val2, y_val2), use_best_model=True)

val_pred2 = model2.predict(X_val2)
val_mae2 = mean_absolute_error(y_val2, val_pred2)
val_r22 = r2_score(y_val2, val_pred2)
print(f"\nWITH CHANNEL FEATURES: MAE={val_mae2:.3f}, R2={val_r22:.3f}")
print(f"Improvement over baseline: {(1 - val_mae2/1.130)*100:.1f}% lower MAE")

importances2 = pd.Series(model2.feature_importances_, index=X_v2.columns).sort_values(ascending=False)
print("\nTop 15 features (with channel stats):")
print(importances2.head(15))

Channel stats computed for 786 channels
       channel_video_count  channel_median_views  channel_view_std  \
count           786.000000          7.860000e+02      7.590000e+02   
mean             28.788804          6.917216e+03      1.324159e+04   
std              19.244949          6.060352e+04      7.048168e+04   
min               1.000000          1.000000e+00      0.000000e+00   
25%               9.000000          7.700000e+01      3.330934e+02   
50%              29.000000          3.710000e+02      7.709321e+02   
75%              50.000000          1.603125e+03      5.076245e+03   
max              50.000000          1.252726e+06      1.135790e+06   

       channel_log_volatility  channel_category_diversity  
count              786.000000                  786.000000  
mean                 1.250281                    1.232824  
std                  0.621722                    0.591224  
min                  0.000000                    1.000000  
25%                  0.842795

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score
from catboost import CatBoostRegressor, CatBoostClassifier

# ---------------------------------------------------------------
# 0. Rebuild channel stats + feature matrix (self-contained, same
#    approach as the m model, so results are directly comparable)
# ---------------------------------------------------------------
channel_stats = df.groupby('channel_id').agg(
    channel_video_count=('id', 'count'),
    channel_median_views=('day_7_views', 'median'),
    channel_view_std=('day_7_views', 'std'),
).reset_index()
channel_stats['channel_log_volatility'] = (
    df.groupby('channel_id')['day_7_views']
    .apply(lambda x: np.std(np.log(x.clip(lower=1)))).values
)
channel_stats['channel_category_diversity'] = (
    df.groupby('channel_id')['category_id'].nunique().values
)

df_shape = df.merge(channel_stats, on='channel_id', how='left')
channel_feature_cols = ['channel_video_count', 'channel_median_views', 'channel_view_std',
                         'channel_log_volatility', 'channel_category_diversity']
all_feature_cols = [c for c in feature_cols if c in df_shape.columns] + channel_feature_cols

# only train shape prediction on videos where the fitted curve itself was trustworthy
usable = df_shape[df_shape['shape_usable'] == True].copy()
print(f"Videos usable for shape training: {len(usable)} / {len(df_shape)}")
print(usable['shape_form'].value_counts())

X_shape = usable[all_feature_cols].fillna(0)
X_shape = X_shape.loc[:, ~X_shape.columns.duplicated()]
groups_shape = usable['channel_id']

# ---------------------------------------------------------------
# 1. Channel-disjoint split (same logic as the m model)
# ---------------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
train_idx, val_idx = next(gss.split(X_shape, usable['shape_form'], groups_shape))

X_train, X_val = X_shape.iloc[train_idx], X_shape.iloc[val_idx]
usable_train, usable_val = usable.iloc[train_idx], usable.iloc[val_idx]

print(f"Train: {len(X_train)}, Val: {len(X_val)}")

# ---------------------------------------------------------------
# 2. Classify which curve form (power vs logistic)
# ---------------------------------------------------------------
y_form_train = (usable_train['shape_form'] == 'logistic').astype(int)
y_form_val = (usable_val['shape_form'] == 'logistic').astype(int)

clf = CatBoostClassifier(iterations=300, depth=5, learning_rate=0.05,
                          verbose=0, random_state=0)
clf.fit(X_train, y_form_train, eval_set=(X_val, y_form_val))

form_pred = clf.predict(X_val)
form_acc = accuracy_score(y_form_val, form_pred)
baseline_acc = max(y_form_val.mean(), 1 - y_form_val.mean())  # always-predict-majority-class baseline
print(f"\nFORM CLASSIFIER: accuracy={form_acc:.3f} (majority-class baseline={baseline_acc:.3f})")

# ---------------------------------------------------------------
# 3. Regress parameters, separately per form
# ---------------------------------------------------------------
results = {}
for form, params in [('power', ['c', 'theta']), ('logistic', ['k', 't0'])]:
    tr_mask = usable_train['shape_form'] == form
    val_mask = usable_val['shape_form'] == form
    if tr_mask.sum() < 50 or val_mask.sum() < 10:
        print(f"\nSkipping {form}: not enough data ({tr_mask.sum()} train, {val_mask.sum()} val)")
        continue

    print(f"\n--- {form.upper()} form ({tr_mask.sum()} train, {val_mask.sum()} val) ---")
    for param in params:
        reg = CatBoostRegressor(iterations=300, depth=5, learning_rate=0.05,
                                 loss_function='MAE', verbose=0, random_state=0)
        reg.fit(X_train[tr_mask], usable_train.loc[tr_mask, param],
                eval_set=(X_val[val_mask], usable_val.loc[val_mask, param]))
        pred = reg.predict(X_val[val_mask])
        true = usable_val.loc[val_mask, param]
        mae = mean_absolute_error(true, pred)
        r2 = r2_score(true, pred)
        baseline_mae = mean_absolute_error(true, [true.mean()] * len(true))
        print(f"  {param}: MAE={mae:.3f} (mean-baseline MAE={baseline_mae:.3f}), R2={r2:.3f}")
        results[f'{form}_{param}'] = {'mae': mae, 'r2': r2, 'baseline_mae': baseline_mae, 'model': reg}

results['form_classifier'] = clf
print("\nDone. Results dict has all fitted models for reuse.")

NameError: name 'feature_cols' is not defined

In [1]:
import pandas as pd
df = pd.read_csv('training_corpus_h7_final.csv')

In [4]:
import pandas as pd
import numpy as np

print("="*60)
print("1. MISSING DATA")
print("="*60)
key_cols = ['title', 'description', 'tags', 'category_id', 'video_duration',
            'thumbnail_default', 'thumbnail_medium', 'channel_id', 'published_at']
for col in key_cols:
    if col in df.columns:
        n_missing = df[col].isna().sum()
        pct = n_missing / len(df) * 100
        print(f"  {col}: {n_missing} missing ({pct:.1f}%)")

print("\n" + "="*60)
print("2. DUPLICATE VIDEOS")
print("="*60)
dupe_ids = df['id'].duplicated().sum()
print(f"  Duplicate video IDs: {dupe_ids}")
if dupe_ids > 0:
    print(df[df['id'].duplicated(keep=False)].sort_values('id')[['id','channel_id','title']].head(10))

print("\n" + "="*60)
print("3. VIEW/LIKE/COMMENT CONSISTENCY")
print("="*60)
# likes and comments should never exceed views by a large margin
if 'day_7_views' in df.columns:
    like_cols = [c for c in df.columns if 'like' in c.lower()]
    if like_cols:
        sample_like_col = 'day_7_likes' if 'day_7_likes' in df.columns else like_cols[0]
        impossible = (df[sample_like_col] > df['day_7_views']).sum()
        print(f"  Videos with day_7_likes > day_7_views (impossible): {impossible}")

print("\n" + "="*60)
print("4. DURATION SANITY")
print("="*60)
if 'duration_s' in df.columns:
    print(df['duration_s'].describe())
    zero_dur = (df['duration_s'] == 0).sum()
    huge_dur = (df['duration_s'] > 3600 * 5).sum()  # >5 hours
    print(f"  Zero duration: {zero_dur}")
    print(f"  Duration > 5 hours (suspicious for typical content): {huge_dur}")

print("\n" + "="*60)
print("5. CATEGORY DISTRIBUTION")
print("="*60)
print(df['category_id'].value_counts().head(15))
# YouTube category IDs are a known fixed set (1-44ish, with gaps).
# Anything wildly outside that range would indicate corrupted data.
valid_range = df['category_id'].between(1, 44)
print(f"\n  Category IDs outside expected range [1,44]: {(~valid_range).sum()}")

print("\n" + "="*60)
print("6. PUBLISH DATE SANITY")
print("="*60)
df['published_at'] = pd.to_datetime(df['published_at'])
now = pd.Timestamp.now(tz=df['published_at'].dt.tz)
future_videos = (df['published_at'] > now).sum()
print(f"  Videos with a publish date in the future: {future_videos}")
print(f"  Earliest: {df['published_at'].min()}")
print(f"  Latest:   {df['published_at'].max()}")

print("\n" + "="*60)
print("7. TITLE/TEXT QUALITY")
print("="*60)
empty_titles = df['title'].fillna('').str.strip().eq('').sum()
very_short_titles = (df['title'].fillna('').str.len() < 3).sum()
print(f"  Empty titles: {empty_titles}")
print(f"  Titles under 3 characters: {very_short_titles}")
print(f"  Title length distribution:\n{df['title'].fillna('').str.len().describe()}")

print("\n" + "="*60)
print("8. THUMBNAIL COVERAGE")
print("="*60)
if 'has_thumbnail' in df.columns:
    print(df['has_thumbnail'].value_counts())

print("\n" + "="*60)
print("9. LABEL SANITY (log_m, shape)")
print("="*60)
print("log_m distribution:")
print(df['log_m_clipped'].describe())
print(f"\nShould be roughly centered near 0 (median video ~ channel typical).")
print(f"Skew: {df['log_m_clipped'].skew():.3f} (near 0 = symmetric, good)")

if 'shape_r2' in df.columns:
    print(f"\nShape fit R2 - median: {df['shape_r2'].median():.3f}, "
          f"% usable (>=0.8): {(df['shape_r2']>=0.8).mean()*100:.1f}%")

print("\n" + "="*60)
print("10. CHANNEL CONCENTRATION")
print("="*60)
per_channel = df.groupby('channel_id').size().sort_values(ascending=False)
print(f"  Total channels: {len(per_channel)}")
print(f"  Top channel's share: {per_channel.iloc[0]/len(df)*100:.2f}%")
print(f"  Top 5 channels' combined share: {per_channel.head(5).sum()/len(df)*100:.2f}%")

print("\n" + "="*60)
print("11. LANGUAGE / SCRIPT SPREAD (rough check)")
print("="*60)
# rough heuristic: does title contain non-Latin script characters?
def has_non_latin(s):
    if pd.isna(s):
        return False
    return any(ord(ch) > 0x0250 for ch in str(s))  # beyond extended Latin
non_latin_share = df['title'].apply(has_non_latin).mean()
print(f"  Titles with likely non-Latin script: {non_latin_share*100:.1f}%")

1. MISSING DATA
  title: 0 missing (0.0%)
  description: 7443 missing (32.9%)
  tags: 0 missing (0.0%)
  category_id: 0 missing (0.0%)
  video_duration: 184 missing (0.8%)
  thumbnail_default: 0 missing (0.0%)
  thumbnail_medium: 0 missing (0.0%)
  channel_id: 0 missing (0.0%)
  published_at: 0 missing (0.0%)

2. DUPLICATE VIDEOS
  Duplicate video IDs: 0

3. VIEW/LIKE/COMMENT CONSISTENCY
  Videos with day_7_likes > day_7_views (impossible): 4

4. DURATION SANITY

5. CATEGORY DISTRIBUTION
category_id
22    5220
27    3732
24    3421
19    2202
28    1650
26    1561
20    1510
2      912
10     880
1      523
17     430
25     424
15     116
29      33
23      14
Name: count, dtype: int64

  Category IDs outside expected range [1,44]: 0

6. PUBLISH DATE SANITY
  Videos with a publish date in the future: 0
  Earliest: 2025-08-07 04:39:33
  Latest:   2026-08-09 05:56:48

7. TITLE/TEXT QUALITY
  Empty titles: 0
  Titles under 3 characters: 31
  Title length distribution:
count    22628.0000

In [8]:
"""
Assumes these files are still on disk from earlier work (same folder as your notebook):
  - training_corpus_h7_final.csv   (labels: log_m, shape params, video metadata)
  - text_embeddings.npy            (512-dim, row-aligned with the CSV)
  - image_embeddings_final.npy     (512-dim, row-aligned; falls back to image_embeddings.npy)

If any of these are missing, they need to be regenerated from the earlier
feature-extraction step -- let me know which one is missing and I'll help
rebuild just that piece rather than everything from scratch.
"""
import numpy as np
import pandas as pd
import re
import json
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score

torch.manual_seed(0)
np.random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# ---------------------------------------------------------------
# 1. Reload the labeled corpus
# ---------------------------------------------------------------
df = pd.read_csv('training_corpus_h7_final.csv')
print(f"Loaded corpus: {len(df)} rows")

# ---------------------------------------------------------------
# 2. Reload embeddings (must match df's row order -- they were saved
#    from this same dataframe earlier, so order is preserved)
# ---------------------------------------------------------------
text_embeddings = np.load('text_embeddings.npy')
try:
    image_embeddings = np.load('image_embeddings_final.npy')
except FileNotFoundError:
    image_embeddings = np.load('image_embeddings.npy')

assert len(text_embeddings) == len(df), \
    f"Mismatch: {len(text_embeddings)} embeddings vs {len(df)} rows -- df was likely filtered after embeddings were saved"
assert len(image_embeddings) == len(df), \
    f"Mismatch: {len(image_embeddings)} embeddings vs {len(df)} rows"
print("Embeddings loaded and row-count matches the corpus.")

# ---------------------------------------------------------------
# 3. Rebuild tabular features (cheap, deterministic, from raw columns)
# ---------------------------------------------------------------
def parse_duration(iso):
    if pd.isna(iso):
        return np.nan
    m = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", str(iso))
    if not m:
        return np.nan
    h, mi, s = (int(x) if x else 0 for x in m.groups())
    return h * 3600 + mi * 60 + s

df['duration_s'] = df['video_duration'].apply(parse_duration)
df['title_length'] = df['title'].fillna('').str.len()
df['description_length'] = df['description'].fillna('').str.len()

def count_tags(t):
    try:
        return len(json.loads(t)) if isinstance(t, str) else 0
    except Exception:
        return 0
df['tag_count'] = df['tags'].apply(count_tags)

df['published_at'] = pd.to_datetime(df['published_at'])
df['publish_hour_sin'] = np.sin(2 * np.pi * df['published_at'].dt.hour / 24)
df['publish_hour_cos'] = np.cos(2 * np.pi * df['published_at'].dt.hour / 24)
df['publish_dow_sin'] = np.sin(2 * np.pi * df['published_at'].dt.dayofweek / 7)
df['publish_dow_cos'] = np.cos(2 * np.pi * df['published_at'].dt.dayofweek / 7)

# has_thumbnail: reconstruct from the embeddings themselves (a real thumbnail
# embedding is never all-zero; the placeholder-filled ones are the mean vector,
# not zero, so check against the original zero-vector convention instead)
row_norm = np.linalg.norm(image_embeddings, axis=1)
df['has_thumbnail'] = row_norm > 1e-6  # true for anything non-trivial

# thumbnail-title alignment (cosine similarity, same as before)
def cosine_sim_batch(a, b):
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
    return np.sum(a_norm * b_norm, axis=1)
df['thumbnail_title_alignment'] = cosine_sim_batch(image_embeddings, text_embeddings)

print("Tabular features rebuilt.")

# ---------------------------------------------------------------
# 4. Rebuild PCA compression of the embeddings (deterministic, same seed)
# ---------------------------------------------------------------
N_COMPONENTS = 32
text_pca = PCA(n_components=N_COMPONENTS, random_state=0).fit_transform(text_embeddings)
img_pca = PCA(n_components=N_COMPONENTS, random_state=0).fit_transform(image_embeddings)
text_cols = [f'text_pc{i}' for i in range(N_COMPONENTS)]
img_cols = [f'img_pc{i}' for i in range(N_COMPONENTS)]
df[text_cols] = text_pca
df[img_cols] = img_pca

# ---------------------------------------------------------------
# 5. Channel-level features
# ---------------------------------------------------------------
channel_stats = df.groupby('channel_id').agg(
    channel_video_count=('id', 'count'),
    channel_median_views=('day_7_views', 'median'),
    channel_view_std=('day_7_views', 'std'),
).reset_index()
channel_stats['channel_log_volatility'] = (
    df.groupby('channel_id')['day_7_views']
    .apply(lambda x: np.std(np.log(x.clip(lower=1)))).values
)
channel_stats['channel_category_diversity'] = (
    df.groupby('channel_id')['category_id'].nunique().values
)
df = df.merge(channel_stats, on='channel_id', how='left')
channel_feature_cols = ['channel_video_count', 'channel_median_views', 'channel_view_std',
                         'channel_log_volatility', 'channel_category_diversity']

# ---------------------------------------------------------------
# 6. Category one-hot + final feature matrix
# ---------------------------------------------------------------
df = df.loc[:, ~df.columns.str.startswith('cat_')]  # safe if rerun
cat_dummies = pd.get_dummies(df['category_id'], prefix='cat')
df = pd.concat([df, cat_dummies], axis=1)

feature_cols = (
    ['duration_s', 'title_length', 'description_length', 'tag_count',
     'publish_hour_sin', 'publish_hour_cos', 'publish_dow_sin', 'publish_dow_cos',
     'thumbnail_title_alignment', 'has_thumbnail']
    + text_cols + img_cols + list(cat_dummies.columns) + channel_feature_cols
)

X = df[feature_cols].fillna(0)
X = X.loc[:, ~X.columns.duplicated()]
y = df['log_m_clipped']
groups = df['channel_id']
print(f"Feature matrix: {X.shape}")

# ---------------------------------------------------------------
# 7. Channel-disjoint split (same logic as before)
# ---------------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=0)
train_idx, temp_idx = next(gss.split(X, y, groups))
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_temp, y_temp, groups_temp = X.iloc[temp_idx], y.iloc[temp_idx], groups.iloc[temp_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=0)
val_idx, test_idx = next(gss2.split(X_temp, y_temp, groups_temp))
X_val, y_val = X_temp.iloc[val_idx], y_temp.iloc[val_idx]

print(f"Train: {len(X_train)}, Val: {len(X_val)}")

# ---------------------------------------------------------------
# 8. Standardize + train the MLP
# ---------------------------------------------------------------
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_val_s = scaler.transform(X_val).astype(np.float32)
y_train_np = y_train.values.astype(np.float32).reshape(-1, 1)
y_val_np = y_val.values.astype(np.float32).reshape(-1, 1)

X_train_t = torch.tensor(X_train_s).to(device)
y_train_t = torch.tensor(y_train_np).to(device)
X_val_t = torch.tensor(X_val_s).to(device)
y_val_t = torch.tensor(y_val_np).to(device)

class MLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x)

model = MLP(X_train_s.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.L1Loss()

BATCH_SIZE = 256
n_train = X_train_t.shape[0]
best_val_mae = float('inf')
best_state = None
patience = 15
patience_counter = 0

for epoch in range(200):
    model.train()
    perm = torch.randperm(n_train)
    epoch_loss = 0.0
    for i in range(0, n_train, BATCH_SIZE):
        idx = perm[i:i+BATCH_SIZE]
        xb, yb = X_train_t[idx], y_train_t[idx]
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    epoch_loss /= n_train

    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t)
        val_mae = loss_fn(val_pred, y_val_t).item()

    if val_mae < best_val_mae - 1e-5:
        best_val_mae = val_mae
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}  train_loss={epoch_loss:.4f}  val_mae={val_mae:.4f}  best={best_val_mae:.4f}")

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    val_pred_final = model(X_val_t).cpu().numpy().flatten()

val_mae_final = mean_absolute_error(y_val, val_pred_final)
val_r2_final = r2_score(y_val, val_pred_final)

print(f"\n{'='*50}")
print(f"BASELINE (assume m=1):  MAE=1.130, R2=0.000")
print(f"CATBOOST:               MAE=1.105, R2=0.045")
print(f"DEEP NN (MLP):          MAE={val_mae_final:.3f}, R2={val_r2_final:.3f}")
print(f"{'='*50}")

Using device: cpu
Loaded corpus: 22628 rows
Embeddings loaded and row-count matches the corpus.
Tabular features rebuilt.
Feature matrix: (22628, 94)
Train: 19620, Val: 1395
Epoch   0  train_loss=1.1866  val_mae=1.1352  best=1.1352
Epoch  10  train_loss=1.1107  val_mae=1.1485  best=1.1339
Early stopping at epoch 17

BASELINE (assume m=1):  MAE=1.130, R2=0.000
CATBOOST:               MAE=1.105, R2=0.045
DEEP NN (MLP):          MAE=1.134, R2=0.001


In [9]:
"""
Assumes these files are still on disk from earlier work (same folder as your notebook):
  - training_corpus_h7_final.csv   (labels: log_m, shape params, video metadata)
  - text_embeddings.npy            (512-dim, row-aligned with the CSV)
  - image_embeddings_final.npy     (512-dim, row-aligned; falls back to image_embeddings.npy)

If any of these are missing, they need to be regenerated from the earlier
feature-extraction step -- let me know which one is missing and I'll help
rebuild just that piece rather than everything from scratch.
"""
import numpy as np
import pandas as pd
import re
import json
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score

torch.manual_seed(0)
np.random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# ---------------------------------------------------------------
# 1. Reload the labeled corpus
# ---------------------------------------------------------------
df = pd.read_csv('training_corpus_h7_final.csv')
print(f"Loaded corpus: {len(df)} rows")

# ---------------------------------------------------------------
# 2. Reload embeddings (must match df's row order -- they were saved
#    from this same dataframe earlier, so order is preserved)
# ---------------------------------------------------------------
text_embeddings = np.load('text_embeddings.npy')
try:
    image_embeddings = np.load('image_embeddings_final.npy')
except FileNotFoundError:
    image_embeddings = np.load('image_embeddings.npy')

assert len(text_embeddings) == len(df), \
    f"Mismatch: {len(text_embeddings)} embeddings vs {len(df)} rows -- df was likely filtered after embeddings were saved"
assert len(image_embeddings) == len(df), \
    f"Mismatch: {len(image_embeddings)} embeddings vs {len(df)} rows"
print("Embeddings loaded and row-count matches the corpus.")

# ---------------------------------------------------------------
# 3. Rebuild tabular features (cheap, deterministic, from raw columns)
# ---------------------------------------------------------------
def parse_duration(iso):
    if pd.isna(iso):
        return np.nan
    m = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", str(iso))
    if not m:
        return np.nan
    h, mi, s = (int(x) if x else 0 for x in m.groups())
    return h * 3600 + mi * 60 + s

df['duration_s'] = df['video_duration'].apply(parse_duration)
df['title_length'] = df['title'].fillna('').str.len()
df['description_length'] = df['description'].fillna('').str.len()

def count_tags(t):
    try:
        return len(json.loads(t)) if isinstance(t, str) else 0
    except Exception:
        return 0
df['tag_count'] = df['tags'].apply(count_tags)

df['published_at'] = pd.to_datetime(df['published_at'])
df['publish_hour_sin'] = np.sin(2 * np.pi * df['published_at'].dt.hour / 24)
df['publish_hour_cos'] = np.cos(2 * np.pi * df['published_at'].dt.hour / 24)
df['publish_dow_sin'] = np.sin(2 * np.pi * df['published_at'].dt.dayofweek / 7)
df['publish_dow_cos'] = np.cos(2 * np.pi * df['published_at'].dt.dayofweek / 7)

# has_thumbnail: reconstruct from the embeddings themselves (a real thumbnail
# embedding is never all-zero; the placeholder-filled ones are the mean vector,
# not zero, so check against the original zero-vector convention instead)
row_norm = np.linalg.norm(image_embeddings, axis=1)
df['has_thumbnail'] = row_norm > 1e-6  # true for anything non-trivial

# thumbnail-title alignment (cosine similarity, same as before)
def cosine_sim_batch(a, b):
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
    return np.sum(a_norm * b_norm, axis=1)
df['thumbnail_title_alignment'] = cosine_sim_batch(image_embeddings, text_embeddings)

print("Tabular features rebuilt.")

# ---------------------------------------------------------------
# 4. Rebuild PCA compression of the embeddings (deterministic, same seed)
# ---------------------------------------------------------------
N_COMPONENTS = 32
text_pca = PCA(n_components=N_COMPONENTS, random_state=0).fit_transform(text_embeddings)
img_pca = PCA(n_components=N_COMPONENTS, random_state=0).fit_transform(image_embeddings)
text_cols = [f'text_pc{i}' for i in range(N_COMPONENTS)]
img_cols = [f'img_pc{i}' for i in range(N_COMPONENTS)]
df[text_cols] = text_pca
df[img_cols] = img_pca

# ---------------------------------------------------------------
# 5. Channel-level features
# ---------------------------------------------------------------
channel_stats = df.groupby('channel_id').agg(
    channel_video_count=('id', 'count'),
    channel_median_views=('day_7_views', 'median'),
    channel_view_std=('day_7_views', 'std'),
).reset_index()
channel_stats['channel_log_volatility'] = (
    df.groupby('channel_id')['day_7_views']
    .apply(lambda x: np.std(np.log(x.clip(lower=1)))).values
)
channel_stats['channel_category_diversity'] = (
    df.groupby('channel_id')['category_id'].nunique().values
)
df = df.merge(channel_stats, on='channel_id', how='left')
channel_feature_cols = ['channel_video_count', 'channel_median_views', 'channel_view_std',
                         'channel_log_volatility', 'channel_category_diversity']

# ---------------------------------------------------------------
# 6. Category one-hot + final feature matrix
# ---------------------------------------------------------------
df = df.loc[:, ~df.columns.str.startswith('cat_')]  # safe if rerun
cat_dummies = pd.get_dummies(df['category_id'], prefix='cat')
df = pd.concat([df, cat_dummies], axis=1)

feature_cols = (
    ['duration_s', 'title_length', 'description_length', 'tag_count',
     'publish_hour_sin', 'publish_hour_cos', 'publish_dow_sin', 'publish_dow_cos',
     'thumbnail_title_alignment', 'has_thumbnail']
    + text_cols + img_cols + list(cat_dummies.columns) + channel_feature_cols
)

X = df[feature_cols].fillna(0)
X = X.loc[:, ~X.columns.duplicated()]
y = df['log_m_clipped']
groups = df['channel_id']
print(f"Feature matrix: {X.shape}")

# ---------------------------------------------------------------
# 7. Channel-disjoint split (same logic as before)
# ---------------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=0)
train_idx, temp_idx = next(gss.split(X, y, groups))
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_temp, y_temp, groups_temp = X.iloc[temp_idx], y.iloc[temp_idx], groups.iloc[temp_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=0)
val_idx, test_idx = next(gss2.split(X_temp, y_temp, groups_temp))
X_val, y_val = X_temp.iloc[val_idx], y_temp.iloc[val_idx]

print(f"Train: {len(X_train)}, Val: {len(X_val)}")

# ---------------------------------------------------------------
# 8. Standardize + train the MLP
# ---------------------------------------------------------------
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_val_s = scaler.transform(X_val).astype(np.float32)
y_train_np = y_train.values.astype(np.float32).reshape(-1, 1)
y_val_np = y_val.values.astype(np.float32).reshape(-1, 1)

X_train_t = torch.tensor(X_train_s).to(device)
y_train_t = torch.tensor(y_train_np).to(device)
X_val_t = torch.tensor(X_val_s).to(device)
y_val_t = torch.tensor(y_val_np).to(device)

class MLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 96), nn.ReLU(), nn.Dropout(0.35),
            nn.Linear(96, 48), nn.ReLU(), nn.Dropout(0.35),
            nn.Linear(48, 16), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(16, 1),
        )
    def forward(self, x):
        return self.net(x)

model = MLP(X_train_s.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=8
)
loss_fn = nn.L1Loss()

BATCH_SIZE = 256
n_train = X_train_t.shape[0]
best_val_mae = float('inf')
best_state = None
patience = 30
patience_counter = 0

for epoch in range(400):
    model.train()
    perm = torch.randperm(n_train)
    epoch_loss = 0.0
    for i in range(0, n_train, BATCH_SIZE):
        idx = perm[i:i+BATCH_SIZE]
        xb, yb = X_train_t[idx], y_train_t[idx]
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    epoch_loss /= n_train

    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t)
        val_mae = loss_fn(val_pred, y_val_t).item()

    scheduler.step(val_mae)

    if val_mae < best_val_mae - 1e-5:
        best_val_mae = val_mae
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 10 == 0:
        lr_now = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:3d}  train_loss={epoch_loss:.4f}  val_mae={val_mae:.4f}  best={best_val_mae:.4f}  lr={lr_now:.6f}")

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    val_pred_final = model(X_val_t).cpu().numpy().flatten()

val_mae_final = mean_absolute_error(y_val, val_pred_final)
val_r2_final = r2_score(y_val, val_pred_final)

print(f"\n{'='*50}")
print(f"BASELINE (assume m=1):  MAE=1.130, R2=0.000")
print(f"CATBOOST:               MAE=1.105, R2=0.045")
print(f"DEEP NN (MLP):          MAE={val_mae_final:.3f}, R2={val_r2_final:.3f}")
print(f"{'='*50}")

Using device: cpu
Loaded corpus: 22628 rows
Embeddings loaded and row-count matches the corpus.
Tabular features rebuilt.
Feature matrix: (22628, 94)
Train: 19620, Val: 1395
Epoch   0  train_loss=1.1578  val_mae=1.1316  best=1.1316  lr=0.000300
Epoch  10  train_loss=1.1397  val_mae=1.1367  best=1.1305  lr=0.000300
Epoch  20  train_loss=1.1216  val_mae=1.1427  best=1.1305  lr=0.000075
Epoch  30  train_loss=1.1131  val_mae=1.1430  best=1.1305  lr=0.000037
Early stopping at epoch 34

BASELINE (assume m=1):  MAE=1.130, R2=0.000
CATBOOST:               MAE=1.105, R2=0.045
DEEP NN (MLP):          MAE=1.130, R2=0.002
